# YemenVision-MTMC — المرحلة ٩: «الدماغ العملاق» للمركبات
**إعادة تعرّف بصري عملاق + فئة دقيقة (تاكسي / هايلكس / باص …) + قراءة اللوحة وتفاصيلها + تتبع متعدد الكاميرات**

هذا الدفتر مستقل ومهيأ لـ Google Colab. عند التشغيل تُزامَن البيانات الكبيرة إلى قرص Colab المحلي القابل للاستئناف، بينما تُحفظ النتائج والنماذج على Drive. لا تُشغّل خلايا التدريب على جهازك المحلي.

تركيز الدفتر بحسب طلبك:

| المحور | ما يفعله الدفتر |
|---|---|
| **ReID عملاق** | بصمة مركبة من عدة هياكل عملاقة مفتوحة المصدر (DINOv2-giant، OpenCLIP bigG، EVA02) مدمجة، تُدرَّب بخسارة ID + Triplet + Circle مع تعزيزات تحاكي تغيّر الزاوية والإضاءة والمسافة، ثم إعادة ترتيب k-reciprocal |
| **فئة المركبة بدقة** | تصنيف هرمي (نوع عام ← فئة دقيقة) على مقصوصاتك المسمّاة، معايرة احتمالات، ورفض «غير معروف» بدل التخمين |
| **اللوحة** | كاشف لوحات + قارئ OCR (EasyOCR جاهز + CRNN-CTC تدرّبه أنت على لوحاتك) + تصويت عبر الإطارات + محلّل لتفاصيل اللوحة اليمنية (المحافظة، الفئة بالرمز واللون) |
| **MTMC** | تتبع لكل كاميرا، بصمة لكل مسار، ثم ربط بين الكاميرات بدمج: المظهر + اللوحة + الفئة + قيد زمني/طوبولوجي |
| **التصدير** | كل النماذج النهائية في ملف/مجلد منفصل على Drive مع `manifest` وبصمات SHA256 ووحدة تشغيل مستقلة |

## قبل أن تبدأ — صراحة ضرورية
1. **«أكبر نموذج» لا يعني تلقائياً «أدق نموذج».** الدقة المتناهية تأتي من **بياناتك المسمّاة** (خاصة طرازات السوق اليمني) أكثر من حجم النموذج. لذلك الدفتر يقيس كل شيء على اختبار نزيه بلا تسرب، ويُعلن `NOT_EVALUATED` حين لا توجد بيانات كافية بدل ادّعاء رقم.
2. **نتيجة فحص التشغيل المرفق:** لا توجد خلية مسجلة كاستثناء Python؛ وصل التنفيذ إلى التصدير والتحقق. لكن معظم مراحل التدريب تم تخطيها بسبب غياب بيانات التدريب. لا يمكن إكمال تدريب حقيقي بلا صور/هويات/لوحات/فيديوهات كافية. أُصلحت في النسخة المعدلة عدة نقاط قد تسبب تعثراً عند توفر البيانات.
3. **الحجم مقابل العتاد:** T4 (16GB) يشغّل هياكل بحجم ViT-L بسلاسة. هياكل bigG/giant تحتاج A100 (40GB) أو ما فوق. الدفتر يختار الطبقة تلقائياً حسب الذاكرة (`TIER`).
4. **الرخص:** Ultralytics YOLO تحت AGPL-3.0، وأوزان DINOv2 / OpenCLIP / EVA لها رخص مستقلة؛ يسجّلها `model_zoo/zoo_manifest.json`. راجعها قبل أي استخدام تجاري.
5. **خصوصية اللوحات:** قراءة اللوحات تعالج بيانات شخصية. فعّل `PRIVACY` في الإعدادات (تشفير/تجزئة اللوحات في السجلات) والتزم بالنظام المحلي.

## البيانات المطلوبة (من Drive)
```
YemenVision_MTMC/
├─ production/                      ← من مراحلك السابقة (detector_best.pt ...)
├─ ReID_uploads/VeRi/               ← VeRi-776 الحقيقية (اختياري لكنه موصى به)
├─ ReID_uploads/VehicleID/          ← اختياري
├─ vehicle_class_crops/<الفئة>/     ← مقصوصات مسمّاة: taxi, hilux, bus_daewoo, minibus ...
├─ yemen_reid_ids/<id>/c<cam>_*.jpg ← (الأهم) مقصوصات لنفس المركبة من كاميراتك: مجلد لكل هوية
├─ plates/images/ + plates/labels.csv ← قصاصات لوحات + نصها (file,text)
├─ plates/yolo/ (images/ labels/ data.yaml) ← اختياري لتدريب كاشف اللوحات
└─ mtmc_videos/<cam_id>.mp4         ← فيديوهات الكاميرات للتشغيل النهائي
```

In [1]:
# ١. الإعداد: Drive + بيئة + اختيار طبقة النموذج حسب GPU
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, math, time, random, shutil, hashlib, subprocess, importlib.util, warnings, copy, itertools, collections
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

def status(kind, msg):
    print({'ok': '✅', 'pause': '⏸️', 'error': '❌', 'warn': '⚠️'}.get(kind, 'ℹ️'), msg)

try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive/YemenVision_MTMC'); LOCAL = Path('/content/stage9_work')
except Exception:
    IN_COLAB = False
    DRIVE_ROOT = Path(os.environ.get('YV_DRIVE_ROOT', './YemenVision_MTMC')); LOCAL = Path(os.environ.get('YV_LOCAL', './stage9_work'))
    status('warn', 'خارج Colab: سيُستخدم مجلد محلي')

RUN_ID   = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR  = DRIVE_ROOT / 'stage9' / RUN_ID
PROD_DIR = DRIVE_ROOT / 'production'
ZOO      = DRIVE_ROOT / 'model_zoo'
FINAL_ROOT = DRIVE_ROOT / 'final_models'          # ← هنا يُحفظ النموذج النهائي في مكان منفصل
for d in (DRIVE_ROOT, LOCAL, RUN_DIR, PROD_DIR, ZOO, FINAL_ROOT): d.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(ZOO / 'hf'); os.environ['TORCH_HOME'] = str(ZOO / 'torch'); os.environ['OPEN_CLIP_CACHE_DIR'] = str(ZOO / 'hf')

need = {'torch': None, 'timm': 'timm', 'open_clip': 'open_clip_torch', 'sklearn': 'scikit-learn', 'scipy': 'scipy', 'cv2': 'opencv-python-headless',
        'ultralytics': 'ultralytics>=8.3.0', 'easyocr': 'easyocr', 'PIL': 'pillow', 'tqdm': 'tqdm', 'matplotlib': 'matplotlib',
        'motmetrics': 'motmetrics', 'rapidfuzz': 'rapidfuzz', 'faiss': 'faiss-cpu', 'transformers': 'transformers', 'safetensors': 'safetensors'}
missing = [pip for mod, pip in need.items() if pip and importlib.util.find_spec(mod) is None]
if missing:
    status('pause', f'تثبيت: {missing}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
    still_missing = [pip for mod, pip in need.items() if pip and importlib.util.find_spec(mod) is None]
    if still_missing:
        raise RuntimeError(f'تعذّر تثبيت الحزم المطلوبة: {still_missing}. افحص اتصال Colab ثم أعد الخلية.')

import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
VALIDATE_ONLY = os.environ.get('YV_VALIDATE_ONLY', '1') != '0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE == 'cuda' else 0.0
SEED = 20260523
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

# الطبقة: S (≤16GB: T4) | L (≥24GB: L4/A10/A5000) | XL (≥38GB: A100) | XXL (≥70GB: A100-80/H100)
TIER = 'XXL' if GPU_GB >= 70 else 'XL' if GPU_GB >= 38 else 'L' if GPU_GB >= 22 else 'S'
if DEVICE == 'cpu' and not VALIDATE_ONLY: raise RuntimeError('لا يوجد GPU. في Colab اختر Runtime → Change runtime type → GPU قبل تشغيل التدريب.')
status('ok', f'GPU={GPU_NAME} ({GPU_GB:.1f} GB) → TIER={TIER} | RUN_ID={RUN_ID}')

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(8 << 20), b''): h.update(c)
    return h.hexdigest()
def atomic_copy(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.tmp'); shutil.copyfile(src, tmp); os.replace(tmp, dst)
def _jd(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def save_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_jd), encoding='utf-8')
def load_json(path, default=None):
    try: return json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception: return default
def newest(globs):
    c = []
    for g in globs: c += list(DRIVE_ROOT.glob(g))
    return max(c, key=lambda p: p.stat().st_mtime) if c else None
def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'),) * 3
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0, c-h), min(1, c+h)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def list_images(root):
    return sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in IMG_EXT) if root and Path(root).exists() else []

# إعداد تحميل البيانات الضخمة إلى قرص Colab المحلي (قابل للاستئناف)
# لا تُشغّل التدريب من Drive مباشرة؛ القراءة من /content أسرع بكثير.
COLAB_DATA_ROOT = LOCAL / 'data'
STAGE_DATA_TO_LOCAL = IN_COLAB and os.environ.get('YV_STAGE_DATA', '1') != '0'
STAGE_VIDEOS = os.environ.get('YV_STAGE_VIDEOS', '1') != '0'
STAGE_DATASET_CANDIDATES = ['ReID_uploads', 'yemen_reid_ids', 'vehicle_class_crops', 'plates', 'mtmc_gt']
if STAGE_VIDEOS:
    STAGE_DATASET_CANDIDATES.append('mtmc_videos')
STAGE_DATASETS = [n for n in STAGE_DATASET_CANDIDATES if (DRIVE_ROOT / n).exists()]
MISSING_DATASETS = [n for n in STAGE_DATASET_CANDIDATES if n not in STAGE_DATASETS]

def _sync_tree(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        status('warn', f'مصدر غير موجود، يُتخطى النسخ: {src}')
        return False
    dst.mkdir(parents=True, exist_ok=True)
    rsync = shutil.which('rsync')
    if rsync:
        r = subprocess.run([rsync, '-a', '--partial', f'{src}/', f'{dst}/'], check=False)
        if r.returncode == 0:
            return True
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return True

def stage_colab_data():
    if not STAGE_DATA_TO_LOCAL:
        return DRIVE_ROOT
    COLAB_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name in STAGE_DATASETS:
        _sync_tree(DRIVE_ROOT / name, COLAB_DATA_ROOT / name)
    status('ok', f'بيانات التدريب المحلية: {COLAB_DATA_ROOT} | المنسوخ: {STAGE_DATASETS}')
    if MISSING_DATASETS:
        status('warn', f'مجلدات غير موجودة في Drive (ستُتخطى بأمان): {MISSING_DATASETS}')
    return COLAB_DATA_ROOT

DATA_ROOT = stage_colab_data()
PROJECT_LAYOUT = {name: str(DATA_ROOT / name) for name in STAGE_DATASETS}
save_json(RUN_DIR / 'data_layout.json', {'drive_root': str(DRIVE_ROOT), 'local_root': str(DATA_ROOT), 'available': STAGE_DATASETS, 'missing': MISSING_DATASETS, 'paths': PROJECT_LAYOUT})


Mounted at /content/drive
⏸️ تثبيت: ['open_clip_torch', 'ultralytics>=8.3.0', 'easyocr', 'motmetrics', 'rapidfuzz', 'faiss-cpu']
✅ GPU=CPU (0.0 GB) → TIER=S | RUN_ID=20261124T192636Z
✅ بيانات التدريب المحلية: /content/stage9_work/data | المنسوخ: ['ReID_uploads']
⚠️ مجلدات غير موجودة في Drive (ستُتخطى بأمان): ['yemen_reid_ids', 'vehicle_class_crops', 'plates', 'mtmc_gt', 'mtmc_videos']


In [2]:
# ٢. الإعدادات — عدّلها قبل التشغيل
CFG = {
  # ---- الهياكل العملاقة (مفتوحة المصدر). تُختار حسب TIER؛ يمكنك فرض اختيارك بتعبئة FORCE_BACKBONES
  'backbones_by_tier': {
     'S':   [{'id': 'dinov2_vitl14', 'kind': 'dinov2', 'res': 224, 'dim': 1024}],
     'L':   [{'id': 'dinov2_vitl14', 'kind': 'dinov2', 'res': 224, 'dim': 1024},
             {'id': 'ViT-L-14|laion2b_s32b_b82k', 'kind': 'openclip', 'res': 224, 'dim': 768}],
     'XL':  [{'id': 'dinov2_vitg14', 'kind': 'dinov2', 'res': 224, 'dim': 1536},
             {'id': 'ViT-bigG-14|laion2b_s39b_b160k', 'kind': 'openclip', 'res': 224, 'dim': 1280}],
     'XXL': [{'id': 'dinov2_vitg14', 'kind': 'dinov2', 'res': 224, 'dim': 1536},
             {'id': 'ViT-bigG-14|laion2b_s39b_b160k', 'kind': 'openclip', 'res': 224, 'dim': 1280},
             {'id': 'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k', 'kind': 'timm', 'res': 448, 'dim': 1024}],
  },
  'FORCE_BACKBONES': None,        # مثال: [{'id':'dinov2_vitg14','kind':'dinov2','res':224,'dim':1536}]
  'fp16': True,
  'feat_cache_dir': str(LOCAL / 'feat_cache'),

  # ---- ReID
  'reid_embed_dim': 768,          # بصمة نهائية (تُطبَّع L2)
  'reid_views_per_image': 6,      # عدد النسخ المعزَّزة لكل صورة عند بناء ذاكرة الميزات
  'reid_epochs': 120, 'reid_P': 24, 'reid_K': 4, 'reid_lr': 3e-4, 'reid_wd': 5e-4,
  'triplet_margin': 0.3, 'circle_m': 0.25, 'circle_gamma': 64, 'label_smooth': 0.1,
  'rerank': {'k1': 20, 'k2': 6, 'lambda': 0.3},

  # ---- فئات المركبة
  'veh_epochs': 100, 'veh_lr': 1e-3, 'veh_wd': 1e-2, 'veh_min_per_class': 8, 'veh_warn_per_class': 30,
  'veh_unknown_target_precision': 0.97,   # دقة مطلوبة عند الإجابة؛ ما دونها يُعاد «غير معروف»
  'veh_test_frac': 0.2, 'veh_val_frac': 0.15,

  # ---- اللوحات
  'plate_det_weights': 'yolo11m.pt',      # يُدرَّب على plates/yolo إن وُجدت
  'plate_det_epochs': 150, 'plate_det_imgsz': 960,
  'plate_img_h': 48, 'plate_img_w': 192, 'plate_ocr_epochs': 150,
  'plate_langs': ['ar', 'en'],
  'PRIVACY': {'hash_plates': True, 'salt': 'غيّر-هذه-القيمة-قبل-الإنتاج', 'store_raw_plate': False},

  # ---- MTMC
  'REQUIRE_ALL_TRAINING': True,
  'VALIDATE_ONLY': True,  # True لفحص الكود دون Drive/بيانات؛ اجعله False قبل التدريب
  'mtmc': {'w_app': 0.55, 'w_plate': 0.30, 'w_cls': 0.15, 'assoc_thr': 0.42, 'plate_exact_bonus': 0.35,
           'min_tracklet_len': 8, 'max_gap_s_default': 600.0, 'tracker': 'botsort.yaml', 'det_conf': 0.25, 'det_imgsz': 960},
}
BACKBONES = CFG['FORCE_BACKBONES'] or CFG['backbones_by_tier'][TIER]
status('ok', 'الهياكل المختارة: ' + ' + '.join(b['id'] for b in BACKBONES))

# ---- تصنيف الفئات: عدّله ليطابق مجلدات vehicle_class_crops عندك. المفتاح = اسم المجلد، القيمة = النوع العام
VEHICLE_TAXONOMY = {
  'taxi': 'car', 'sedan': 'car', 'hatchback': 'car', 'suv': 'car', 'jeep_landcruiser': 'car',
  'hilux': 'pickup', 'pickup_other': 'pickup', 'dyna_truck': 'truck', 'truck_heavy': 'truck', 'garbage_truck': 'truck',
  'minibus_hiace': 'minibus', 'minibus_other': 'minibus', 'bus_daewoo': 'bus', 'bus_other': 'bus',
  'van': 'van', 'ambulance': 'emergency', 'police_car': 'emergency', 'army_vehicle': 'military',
  'motorbike': 'two_wheeler', 'tuk_tuk': 'three_wheeler',
}
# ---- اللوحة اليمنية: المحافظات (للتحليل) — فئة اللوحة تُضبط من عيّناتك (انظر الخلية ١٢)
YEMEN_GOVERNORATES_AR = ['أمانة العاصمة','صنعاء','عدن','تعز','إب','الحديدة','حضرموت','ذمار','حجة','المحويت','عمران','صعدة','البيضاء','مأرب','الجوف','لحج','أبين','شبوة','المهرة','الضالع','ريمة','سقطرى']
# ---- طوبولوجيا الكاميرات: زمن السفر الأدنى/الأقصى بالثواني بين كاميرتين (اتركها فارغة = بلا قيد زمني صارم)
CAMERA_TOPOLOGY = {   # ('cam01','cam02'): (min_s, max_s)
}

✅ الهياكل المختارة: dinov2_vitl14


## ٣. جرد البيانات وفحص سلامتها
الجرد يخبرك بدقة ما هو متاح وما ينقص. أي مصدر غير موجود يُتخطى بإعلان صريح، ولا يُخترع له رقم.

In [3]:
# ٣. جرد البيانات + فحص الصور التالفة والمكررة حرفياً
VERI_ROOT = DATA_ROOT / 'ReID_uploads' / 'VeRi'
YEM_ID_DIR = DATA_ROOT / 'yemen_reid_ids'
VEH_CROPS = DATA_ROOT / 'vehicle_class_crops'
PLATE_DIR = DATA_ROOT / 'plates'
VIDEO_DIR = DATA_ROOT / 'mtmc_videos'
VERI_RE = re.compile(r'(\d{4})_c(\d{3})_(\d+)_(\d+)\.(?:jpg|jpeg|png)', re.I)
YEM_RE  = re.compile(r'c(\d+)', re.I)

def check_images(paths, label):
    bad, seen, dup = [], {}, []
    for p in tqdm(paths, desc=f'فحص {label}', leave=False):
        try:
            with Image.open(p) as im: im.verify()
            h = sha256_file(p)
            if h in seen: dup.append((str(p), str(seen[h])))
            else: seen[h] = p
        except Exception:
            bad.append(str(p))
    return bad, dup

INV = {}
veri_imgs = list_images(VERI_ROOT)
INV['veri_images'] = len(veri_imgs)
INV['yemen_reid_images'] = len(list_images(YEM_ID_DIR))
INV['yemen_reid_ids'] = len([d for d in YEM_ID_DIR.iterdir() if d.is_dir()]) if YEM_ID_DIR.exists() else 0
cls_counts = {d.name: len(list_images(d)) for d in sorted(VEH_CROPS.iterdir()) if d.is_dir()} if VEH_CROPS.exists() else {}
INV['vehicle_class_counts'] = cls_counts
INV['plate_label_rows'] = len(pd.read_csv(PLATE_DIR / 'labels.csv')) if (PLATE_DIR / 'labels.csv').is_file() else 0
INV['plate_yolo_yaml'] = (PLATE_DIR / 'yolo' / 'data.yaml').is_file()
INV['videos'] = sorted(p.name for p in VIDEO_DIR.glob('*.mp4')) if VIDEO_DIR.exists() else []
print(json.dumps(INV, ensure_ascii=False, indent=2))

for name, root in (('yemen_reid_ids', YEM_ID_DIR), ('vehicle_class_crops', VEH_CROPS)):
    imgs = list_images(root)
    if imgs:
        bad, dup = check_images(imgs, name)
        status('warn' if (bad or dup) else 'ok', f'{name}: {len(imgs)} صورة | تالف={len(bad)} | مكرر حرفياً={len(dup)}')
        if bad: save_json(RUN_DIR / f'bad_{name}.json', bad)
        if dup: save_json(RUN_DIR / f'dup_{name}.json', dup)
weak = {k: v for k, v in cls_counts.items() if v < CFG['veh_warn_per_class']}
if weak: status('warn', f'فئات بصور قليلة (<{CFG["veh_warn_per_class"]}): {weak} — أضف صوراً مسمّاة قبل الاعتماد على دقتها')
unk = [k for k in cls_counts if k not in VEHICLE_TAXONOMY]
if unk: status('warn', f'مجلدات غير موجودة في VEHICLE_TAXONOMY (ستُعامل كنوع "other"): {unk}')
save_json(RUN_DIR / 'inventory.json', INV)

# فحص صارم قبل التدريب: لا يسمح بتخطي أي مرحلة بصمت
TRAINING_PREFLIGHT_ERRORS = []
if not (VERI_ROOT.exists() or YEM_ID_DIR.exists()): TRAINING_PREFLIGHT_ERRORS.append('ReID: أضف VeRi كاملة داخل ReID_uploads/VeRi أو yemen_reid_ids/<id>')
if len(cls_counts) < 2: TRAINING_PREFLIGHT_ERRORS.append('تصنيف المركبات: يلزم vehicle_class_crops وفيه فئتان على الأقل')
if INV['plate_label_rows'] < 50: TRAINING_PREFLIGHT_ERRORS.append('OCR اللوحات: يلزم plates/labels.csv مع 50 صفًا صحيحًا على الأقل')
if not INV['plate_yolo_yaml'] and not (PROD_DIR / 'plate_best.pt').is_file(): TRAINING_PREFLIGHT_ERRORS.append('كشف اللوحات: أضف plates/yolo/data.yaml أو production/plate_best.pt')
if CFG['REQUIRE_ALL_TRAINING'] and not CFG['VALIDATE_ONLY'] and TRAINING_PREFLIGHT_ERRORS:
    save_json(RUN_DIR / 'training_preflight_failed.json', {'errors': TRAINING_PREFLIGHT_ERRORS, 'inventory': INV, 'layout': PROJECT_LAYOUT})
    raise RuntimeError('فشل فحص جاهزية التدريب الكامل:\n- ' + '\n- '.join(TRAINING_PREFLIGHT_ERRORS))
if CFG['VALIDATE_ONLY']:
    status('warn', 'وضع الفحص فقط: لم يبدأ التدريب ولم تُطلب بيانات Drive')
else:
    status('ok', 'فحص جاهزية التدريب الكامل نجح')


{
  "veri_images": 10,
  "yemen_reid_images": 0,
  "yemen_reid_ids": 0,
  "vehicle_class_counts": {},
  "plate_label_rows": 0,
  "plate_yolo_yaml": false,
  "videos": []
}
⚠️ وضع الفحص فقط: لم يبدأ التدريب ولم تُطلب بيانات Drive


## تنبيه مهم بعد فحص الملف المرفق
المخرجات المحفوظة في هذه النسخة لا تحتوي على استثناء Python؛ التنفيذ وصل إلى التصدير. لكن التدريب الفعلي لم يبدأ لأن مجلدات بيانات ReID والفئات واللوحات والفيديوهات كانت فارغة تقريباً. وجود 10 صور VeRi في الجذر لا يكفي، والدفتر يحتاج بنية VeRi الصحيحة (`image_train/`, `image_query/`, `image_test/`) أو هويات يمنية منظمة. لا يمكن إصلاح نقص البيانات برمجياً أو توليد نتائج تدريب حقيقية منها.

بعد إضافة البيانات، أعد تشغيل الدفتر من البداية. الخلايا التي تتخطى التدريب ستعلن `NOT_EVALUATED` بدلاً من اختلاق مقاييس.

## ٤. المُرمِّز العملاق (Giant Encoder)
يدمج عدة هياكل مفتوحة المصدر: كل هيكل يُطبَّع L2 ثم تُلصق البصمات. هذا الدمج يعطي عادةً تعميماً أفضل من هيكل واحد لأن DINOv2 يلتقط البنية والشكل الهندسي، وOpenCLIP يلتقط الدلالة (ماركة/طراز/لون).

In [4]:
# ٤. المُرمِّز العملاق: DINOv2 + OpenCLIP + (EVA02) — تحميل من المصادر المفتوحة
import torchvision.transforms as T

CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
IN_MEAN, IN_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class _One(nn.Module):
    def __init__(self, spec, device, fp16):
        super().__init__()
        self.spec, self.device = spec, device
        self.dtype = torch.float16 if (fp16 and device == 'cuda') else torch.float32
        k, mid = spec['kind'], spec['id']
        if k == 'dinov2':
            self.m = torch.hub.load('facebookresearch/dinov2', mid)
            self.mean, self.std = IN_MEAN, IN_STD
        elif k == 'openclip':
            import open_clip
            name, tag = mid.split('|')
            self.m, _, _ = open_clip.create_model_and_transforms(name, pretrained=tag)
            self.mean, self.std = CLIP_MEAN, CLIP_STD
        elif k == 'timm':
            import timm
            self.m = timm.create_model(mid, pretrained=True, num_classes=0)
            cfgm = timm.data.resolve_data_config({}, model=self.m)
            self.mean, self.std = cfgm['mean'], cfgm['std']
            spec['res'] = int(cfgm['input_size'][-1])
        else:
            raise ValueError(k)
        self.m = self.m.to(device).to(self.dtype).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.res = int(spec['res'])
        self.register_buffer('mu', torch.tensor(self.mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('sd', torch.tensor(self.std).view(1, 3, 1, 1), persistent=False)
        self.to(device)
    def forward(self, x):   # x: float [0,1] (B,3,res,res)
        x = ((x - self.mu) / self.sd).to(self.dtype)
        k = self.spec['kind']
        if k == 'dinov2':
            o = self.m.forward_features(x)
            f = torch.cat([o['x_norm_clstoken'], o['x_norm_patchtokens'].mean(1)], dim=1)
        elif k == 'openclip':
            f = self.m.encode_image(x)
        else:
            f = self.m(x)
        return F.normalize(f.float(), dim=1)

class GiantEncoder:
    def __init__(self, specs, device=DEVICE, fp16=CFG['fp16']):
        self.parts = []
        for s in specs:
            t0 = time.time()
            try:
                self.parts.append(_One(copy.deepcopy(s), device, fp16)); status('ok', f'حُمّل {s["id"]} ({time.time()-t0:.0f}s)')
            except Exception as e:
                status('error', f'تعذّر تحميل {s["id"]}: {str(e)[:200]} — يُتخطى')
        assert self.parts, 'لا يوجد أي هيكل صالح'
        self.device = device
        with torch.no_grad():
            self.dim = sum(p(torch.zeros(1, 3, p.res, p.res, device=device)).shape[1] for p in self.parts)
        self.bs = {'S': 48, 'L': 96, 'XL': 192, 'XXL': 256}[TIER]
        status('ok', f'بُعد البصمة المدمجة = {self.dim}')
    @torch.no_grad()
    def encode_pils(self, pils):
        outs = []
        for p in self.parts:
            arr = np.stack([np.asarray(im.convert('RGB').resize((p.res, p.res), Image.BICUBIC)) for im in pils])
            x = torch.from_numpy(arr).to(self.device).permute(0, 3, 1, 2).float() / 255.0
            outs.append(p(x))
        return torch.cat(outs, 1).cpu().numpy()
    def encode_many(self, pils):
        res = [self.encode_pils(pils[i:i+self.bs]) for i in range(0, len(pils), self.bs)]
        return np.concatenate(res, 0) if res else np.zeros((0, self.dim), np.float32)

ENC = GiantEncoder(BACKBONES)
manifest = {'backbones': [p.spec for p in ENC.parts], 'dim': ENC.dim, 'tier': TIER, 'gpu': GPU_NAME,
            'licenses_note': 'DINOv2: Apache-2.0 | OpenCLIP LAION weights: MIT | EVA02/timm: حسب بطاقة كل نموذج | Ultralytics: AGPL-3.0'}
save_json(ZOO / 'zoo_manifest.json', manifest)

Using cache found in /content/drive/MyDrive/YemenVision_MTMC/model_zoo/torch/hub/facebookresearch_dinov2_main


✅ حُمّل dinov2_vitl14 (56s)
✅ بُعد البصمة المدمجة = 2048


In [5]:
# ٥. تعزيزات تحاكي واقع الكاميرات (زاوية/مسافة/إضاءة/ضبابية/حجب) + بناء ذاكرة الميزات
from concurrent.futures import ThreadPoolExecutor

def _lowres(im):
    w, h = im.size; s = random.uniform(0.25, 0.6)
    return im.resize((max(8, int(w*s)), max(8, int(h*s))), Image.BILINEAR).resize((w, h), Image.BICUBIC)
def _erase(im, p=0.35):
    if random.random() > p: return im
    im = im.copy(); w, h = im.size
    ew, eh = int(w*random.uniform(0.1, 0.35)), int(h*random.uniform(0.1, 0.35))
    x0, y0 = random.randint(0, max(0, w-ew)), random.randint(0, max(0, h-eh))
    im.paste(tuple(random.randint(0, 255) for _ in range(3)), (x0, y0, x0+ew, y0+eh)); return im
AUG = T.Compose([
    T.RandomResizedCrop((256, 256), scale=(0.55, 1.0), ratio=(0.7, 1.4)),
    T.RandomPerspective(0.35, p=0.6),
    T.RandomRotation(8),
    T.ColorJitter(0.45, 0.45, 0.45, 0.04),
    T.RandomHorizontalFlip(0.5),
    T.RandomApply([T.Lambda(_lowres)], p=0.4),
    T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.2),
    T.RandomGrayscale(0.05),
    T.Lambda(_erase),
])

def _load(p):
    try: return Image.open(p).convert('RGB')
    except Exception: return Image.new('RGB', (128, 128))

def build_feature_cache(paths, views, tag):
    """يعيد مصفوفة (N, V, D) float16. الإطار 0 نظيف دائماً. يُخزَّن على القرص ويُستأنف عند الانقطاع."""
    out_dir = Path(CFG['feat_cache_dir']); out_dir.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha1((tag + str(views) + str(ENC.dim) + '|'.join(map(str, paths))).encode()).hexdigest()[:16]
    f = out_dir / f'{tag}_{key}.npy'
    if f.is_file(): status('ok', f'ذاكرة ميزات جاهزة: {f.name}'); return np.load(f)
    N = len(paths); tmp = f.with_name(f.name + '.partial')
    if tmp.exists(): tmp.unlink()
    arr = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float16, shape=(N, views, ENC.dim))
    B = ENC.bs
    with ThreadPoolExecutor(max_workers=min(4, os.cpu_count() or 1)) as ex:
        for i in tqdm(range(0, N, B), desc=f'ميزات {tag}'):
            imgs = list(ex.map(_load, paths[i:i+B]))
            for v in range(views):
                batch = imgs if v == 0 else [AUG(im) for im in imgs]
                arr[i:i+len(imgs), v] = ENC.encode_pils(batch).astype(np.float16)
            arr.flush()
    del arr
    os.replace(tmp, f)
    return np.load(f, mmap_mode='r')

## ٦. فئات المركبة بدقة (تاكسي / هايلكس / باص دايو / …)
**التقسيم بلا تسرب:** الصور المتشابهة جداً (إطارات متتالية لنفس المركبة) تُجمَّع بـ dHash ثم تُقسَّم **كمجموعات كاملة**، فلا تدخل صورة قريبة من صورة الاختبار إلى التدريب.
**الرفض:** يُحسب حد ثقة على مجموعة التحقق يضمن الدقة المطلوبة عند الإجابة؛ ما دونه يُعاد «غير معروف».

In [6]:
# ٦. بناء مجموعة الفئات + تقسيم بلا تسرب بمجموعات شبه-المكررات
POP = np.array([bin(i).count('1') for i in range(256)], np.uint8)
def dhash_bits(im):
    g = np.asarray(im.convert('L').resize((9, 8), Image.BILINEAR), np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten())
def near_dup_groups(paths, thr=6):
    P = np.stack([dhash_bits(_load(p)) for p in paths]); n = len(paths); par = list(range(n))
    def find(a):
        while par[a] != a: par[a] = par[par[a]]; a = par[a]
        return a
    for i in range(n - 1):
        d = POP[np.bitwise_xor(P[i], P[i+1:])].sum(1)
        for j in np.nonzero(d <= thr)[0]:
            ra, rb = find(i), find(i + 1 + int(j))
            if ra != rb: par[rb] = ra
    return [find(i) for i in range(n)]

def build_vehicle_table():
    rows = []
    for cdir in sorted(d for d in VEH_CROPS.iterdir() if d.is_dir()) if VEH_CROPS.exists() else []:
        ps = list_images(cdir)
        if len(ps) < CFG['veh_min_per_class']:
            status('warn', f'{cdir.name}: {len(ps)} صورة < الحد الأدنى {CFG["veh_min_per_class"]} → NOT_EVALUATED'); continue
        grp = near_dup_groups(ps)
        rng = random.Random(SEED); uniq = sorted(set(grp)); rng.shuffle(uniq)
        ng = len(uniq); n_test = max(1, int(round(ng*CFG['veh_test_frac']))) if ng >= 3 else 0
        n_val = max(1, int(round(ng*CFG['veh_val_frac']))) if ng >= 4 else 0
        assign = {g: ('test' if k < n_test else 'val' if k < n_test + n_val else 'train') for k, g in enumerate(uniq)}
        for p, g in zip(ps, grp): rows.append({'path': str(p), 'cls': cdir.name, 'coarse': VEHICLE_TAXONOMY.get(cdir.name, 'other'), 'group': f'{cdir.name}:{g}', 'split': assign[g]})
    return pd.DataFrame(rows)

VEH_DF = build_vehicle_table()
_veh_train_counts = VEH_DF[VEH_DF.split == 'train'].groupby('cls').size() if len(VEH_DF) else pd.Series(dtype=int)
HAVE_VEH = (len(VEH_DF) > 0 and VEH_DF['cls'].nunique() >= 2
            and (VEH_DF.split == 'test').sum() > 0
            and (VEH_DF.split == 'train').sum() >= 2
            and (_veh_train_counts >= 2).sum() >= 2)
if HAVE_VEH:
    print(VEH_DF.pivot_table(index='cls', columns='split', values='path', aggfunc='count', fill_value=0))
    leak = set(VEH_DF[VEH_DF.split == 'train'].group) & set(VEH_DF[VEH_DF.split == 'test'].group)
    assert not leak, 'تسرب بين التدريب والاختبار!'
    VEH_DF.to_csv(RUN_DIR / 'vehicle_split.csv', index=False)
else:
    status('pause', 'لا توجد بيانات فئات كافية (فئتان على الأقل مع عينتين تدريبيتين لكل فئة واختبار): ضع مقصوصات في vehicle_class_crops/<الفئة>/ ثم أعد الخلية. (يُتخطى هذا القسم)')

⏸️ لا توجد بيانات فئات كافية (فئتان على الأقل مع عينتين تدريبيتين لكل فئة واختبار): ضع مقصوصات في vehicle_class_crops/<الفئة>/ ثم أعد الخلية. (يُتخطى هذا القسم)


In [7]:
# ٧. تدريب مصنّف الفئات على البصمة العملاقة + معايرة + رفض «غير معروف» + تقييم نزيه
VEH_ART = None
class VehHead(nn.Module):
    def __init__(self, d, c, h=1024, p=0.3):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, c))
    def forward(self, x): return self.net(x)

if HAVE_VEH:
    classes = sorted(VEH_DF.cls.unique()); c2i = {c: i for i, c in enumerate(classes)}
    coarse_names = sorted(set(VEH_DF.coarse)); co2i = {c: i for i, c in enumerate(coarse_names)}
    fine_to_coarse = torch.tensor([co2i[VEH_DF[VEH_DF.cls == c].coarse.iloc[0]] for c in classes])
    parts = {s: VEH_DF[VEH_DF.split == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
    feats = {'train': build_feature_cache(list(parts['train'].path), 5, 'veh_train')}
    for s in ('val', 'test'):
        if len(parts[s]): feats[s] = build_feature_cache(list(parts[s].path), 1, f'veh_{s}')[:, 0].astype(np.float32)
    ytr = np.array([c2i[c] for c in parts['train'].cls])
    freq = np.bincount(ytr, minlength=len(classes)).astype(np.float32); w = torch.tensor((freq.max() / np.maximum(freq, 1)) ** 0.5, device=DEVICE)
    head = VehHead(ENC.dim, len(classes)).to(DEVICE)
    opt = torch.optim.AdamW(head.parameters(), lr=CFG['veh_lr'], weight_decay=CFG['veh_wd'])
    E = CFG['veh_epochs']; sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['veh_lr'], total_steps=E * math.ceil(len(ytr) / 128))
    Xtr = torch.from_numpy(feats['train']); Ytr = torch.from_numpy(ytr)
    def logits_of(X):
        head.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), 512): out.append(head(torch.as_tensor(X[i:i+512], device=DEVICE).float()).cpu())
        return torch.cat(out)
    best, best_state, bad = -1, None, 0
    for ep in range(E):
        head.train(); perm = torch.randperm(len(ytr))
        for i in range(0, len(perm), 128):
            idx = perm[i:i+128]; v = torch.randint(0, Xtr.shape[1], (len(idx),))
            x = Xtr[idx, v].to(DEVICE).float(); y = Ytr[idx].to(DEVICE)
            x = x + 0.01 * torch.randn_like(x)
            loss = F.cross_entropy(head(x), y, weight=w, label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if 'val' in feats:
            acc = (logits_of(feats['val']).argmax(1).numpy() == np.array([c2i[c] for c in parts['val'].cls])).mean()
            if acc > best: best, best_state, bad = acc, copy.deepcopy(head.state_dict()), 0
            else: bad += 1
            if bad >= 8: break
    if best_state: head.load_state_dict(best_state)
    status('ok', f'انتهى التدريب | أفضل دقة تحقق={best:.3f}')

    # --- معايرة الحرارة + حد الرفض على التحقق
    T_ = 1.0; thr = 0.5
    if 'val' in feats:
        lv = logits_of(feats['val']); yv = torch.tensor([c2i[c] for c in parts['val'].cls]); logT = torch.zeros(1, requires_grad=True)
        o = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)
        def cl(): o.zero_grad(); l = F.cross_entropy(lv / logT.exp(), yv); l.backward(); return l
        o.step(cl); T_ = float(logT.exp())
        pv = F.softmax(lv / T_, 1); conf, pred = pv.max(1); ok = (pred == yv).numpy(); cf = conf.numpy()
        cand = None
        for t in np.unique(np.round(cf, 4)):
            m = cf >= t
            if m.sum() >= max(5, 0.2 * len(cf)) and ok[m].mean() >= CFG['veh_unknown_target_precision']: cand = float(t); break
        if cand is None: status('warn', 'لا يوجد حد يحقق الدقة المطلوبة بتغطية ≥20% على التحقق → استُعمل 0.5 (دقة أقل من المطلوب)')
        thr = cand if cand is not None else 0.5
    status('ok', f'T={T_:.3f} | حد الثقة للرفض={thr:.3f}')

    # --- تقييم الاختبار (لم يُستعمل في أي قرار)
    if 'test' in feats:
        pt = F.softmax(logits_of(feats['test']) / T_, 1); conf, pred = pt.max(1); yt = np.array([c2i[c] for c in parts['test'].cls])
        pred, conf = pred.numpy(), conf.numpy(); ans = conf >= thr
        co_ok = (fine_to_coarse[pred].numpy() == fine_to_coarse[yt].numpy())
        rep = {'n_test': int(len(yt)), 'top1_all': wilson(int((pred == yt).sum()), len(yt)), 'coarse_acc': wilson(int(co_ok.sum()), len(yt)),
               'coverage': float(ans.mean()), 'acc_when_answered': wilson(int((pred[ans] == yt[ans]).sum()), int(ans.sum())) if ans.any() else None, 'per_class': {}}
        for c, i in c2i.items():
            m = yt == i
            rep['per_class'][c] = {'n_test': int(m.sum()), 'recall': wilson(int((pred[m] == i).sum()), int(m.sum())) if m.sum() else 'NOT_EVALUATED',
                                   'precision': (float(((pred == i) & (yt == i)).sum() / max(1, (pred == i).sum())))}
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(yt, pred, labels=range(len(classes)))
        pairs = sorted([(int(cm[i, j]), classes[i], classes[j]) for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0], reverse=True)[:12]
        rep['top_confusions(count,true,pred)'] = pairs
        save_json(RUN_DIR / 'vehicle_class_report.json', rep)
        print('Top-1 (p, lo, hi):', [round(x, 3) for x in rep['top1_all']], '| نوع عام:', [round(x, 3) for x in rep['coarse_acc']], '| تغطية:', round(rep['coverage'], 3))
        for c, r in rep['per_class'].items(): print(f'  {c:18s} n={r["n_test"]:4d} recall={r["recall"] if isinstance(r["recall"], str) else [round(x,3) for x in r["recall"]]}')
        print('أكثر الالتباسات:', pairs[:6])
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(max(6, len(classes)*0.5), max(5, len(classes)*0.45))); ax.imshow(cm, cmap='Blues')
        ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=90); ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes); plt.tight_layout()
        fig.savefig(RUN_DIR / 'vehicle_confusion.png', dpi=130); plt.show()
    VEH_ART = {'classes': classes, 'coarse_of': {c: VEHICLE_TAXONOMY.get(c, 'other') for c in classes}, 'state': {k: v.cpu() for k, v in head.state_dict().items()},
               'in_dim': ENC.dim, 'temperature': T_, 'reject_threshold': thr, 'test_report_path': str(RUN_DIR / 'vehicle_class_report.json')}

## ٨. المنطق المشترك (مصدر واحد للحقيقة)
هذه الخلية تحوي كل المنطق الخالص (محلّل اللوحة، تصويت الإطارات، الربط بين الكاميرات، إعادة الترتيب). **تُنفَّذ هنا وتُكتب حرفياً داخل وحدة التشغيل النهائية** (`yemenvision_stage9_runtime.py`)، فلا يختلف سلوك الدفتر عن سلوك الإنتاج. اختُبرت محلياً باختبارات دخان قبل تسليمها.

In [8]:
# ٨. المنطق المشترك — يُنفَّذ الآن ويُكتب في وحدة التشغيل النهائية
COMMON_SRC = r"""
import re, math, hashlib
import numpy as np

# ---------------- نصوص عربية ولوحات ----------------
_AR_DIG = {ord(c): str(i) for i, c in enumerate('٠١٢٣٤٥٦٧٨٩')}
_AR_DIG.update({ord(c): str(i) for i, c in enumerate('۰۱۲۳۴۵۶۷۸۹')})
_DIAC = re.compile('[\u064b-\u065f\u0670\u0640]')
def normalize_ar(s):
    s = (s or '').translate(_AR_DIG)
    s = _DIAC.sub('', s)
    s = re.sub('[أإآٱ]', 'ا', s).replace('ى', 'ي').replace('ة', 'ه').replace('ؤ', 'و').replace('ئ', 'ي')
    return re.sub(r'\s+', ' ', s).strip()

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]
def text_sim(a, b):
    m = max(len(a), len(b))
    return 1.0 if m == 0 else 1.0 - levenshtein(a, b) / m

def parse_plate(text, governorates, gov_thr=0.72):
    t = normalize_ar(text)
    digits = re.findall(r'\d+', t)
    letters = re.findall(r'[\u0621-\u064a]+', t)
    number = max(digits, key=len) if digits else ''
    gov, gov_score = None, 0.0
    ng = {g: normalize_ar(g) for g in governorates}
    cands = letters + ([' '.join(letters)] if len(letters) > 1 else [])
    for c in cands:
        for g, gn in ng.items():
            s = text_sim(c, gn)
            if s > gov_score: gov, gov_score = g, s
    if gov_score < gov_thr: gov = None
    rest = [l for l in letters if gov is None or text_sim(l, ng[gov]) < gov_thr]
    return {'raw': text, 'normalized': t, 'number': number, 'all_numbers': digits, 'governorate': gov,
            'governorate_score': round(float(gov_score), 3), 'letters': rest, 'valid': bool(number)}

def hash_plate(text, salt):
    return hashlib.sha256((salt + normalize_ar(text)).encode('utf-8')).hexdigest()[:16]

def vote_plate(reads, min_conf=0.15):
    # reads: [(text, conf)] — تصويت موزون بالثقة على مستوى الحرف بعد اختيار الطول الغالب
    reads = [(normalize_ar(t), float(c)) for t, c in reads if t and c >= min_conf]
    if not reads: return {'text': '', 'conf': 0.0, 'agreement': 0.0, 'n': 0}
    wlen = {}
    for t, c in reads: wlen[len(t)] = wlen.get(len(t), 0.0) + c
    L = max(wlen, key=wlen.get)
    sel = [(t, c) for t, c in reads if len(t) == L]
    out, confs = [], []
    for i in range(L):
        tally = {}
        for t, c in sel: tally[t[i]] = tally.get(t[i], 0.0) + c
        ch = max(tally, key=tally.get); out.append(ch); confs.append(tally[ch] / sum(tally.values()))
    text = ''.join(out)
    agree = sum(1 for t, _ in sel if t == text) / len(sel)
    return {'text': text, 'conf': float(np.mean(confs)) * (len(sel) / len(reads)) ** 0.5, 'agreement': agree, 'n': len(sel)}

# ---------------- إعادة الترتيب k-reciprocal (Zhong et al. 2017) ----------------
def re_ranking(q_g, q_q, g_g, k1=20, k2=6, lam=0.3):
    od = np.concatenate([np.concatenate([q_q, q_g], 1), np.concatenate([q_g.T, g_g], 1)], 0)
    od = np.power(od, 2).astype(np.float32)
    od = np.transpose(1. * od / np.max(od, axis=0))
    V = np.zeros_like(od, dtype=np.float32)
    rank = np.argsort(od).astype(np.int32)
    qn, n = q_g.shape[0], od.shape[0]
    for i in range(n):
        fw = rank[i, :k1 + 1]; bw = rank[fw, :k1 + 1]
        kr = fw[np.where(bw == i)[0]]; kre = kr
        for c in kr:
            cf = rank[c, :int(np.around(k1 / 2)) + 1]; cb = rank[cf, :int(np.around(k1 / 2)) + 1]
            ckr = cf[np.where(cb == c)[0]]
            if len(np.intersect1d(ckr, kr)) > 2 / 3 * len(ckr): kre = np.append(kre, ckr)
        kre = np.unique(kre); w = np.exp(-od[i, kre]); V[i, kre] = w / w.sum()
    od = od[:qn]
    if k2 != 1:
        Vq = np.zeros_like(V)
        for i in range(n): Vq[i] = np.mean(V[rank[i, :k2]], axis=0)
        V = Vq
    inv = [np.where(V[:, i] != 0)[0] for i in range(n)]
    jd = np.zeros_like(od, dtype=np.float32)
    for i in range(qn):
        tm = np.zeros((1, n), np.float32); nz = np.where(V[i] != 0)[0]
        for j in nz: tm[0, inv[j]] += np.minimum(V[i, j], V[inv[j], j])
        jd[i] = 1 - tm / (2. - tm)
    fd = jd * (1 - lam) + od * lam
    return fd[:, qn:]

# ---------------- الربط بين الكاميرات ----------------
def tracklet_pair_cost(a, b, cfg, topology):
    # a,b: dict(cam,t0,t1,emb(L2),coarse,fine,cls_conf,plate,plate_conf)
    if a['cam'] == b['cam']:
        if not (a['t1'] < b['t0'] or b['t1'] < a['t0']): return None          # تداخل زمني في الكاميرا نفسها = مركبتان مختلفتان
    else:
        first, second = (a, b) if a['t0'] <= b['t0'] else (b, a)
        gap = second['t0'] - first['t1']
        rule = topology.get((first['cam'], second['cam'])) or topology.get((second['cam'], first['cam']))
        if rule is not None and not (rule[0] <= gap <= rule[1]): return None
        if rule is None and gap > cfg['max_gap_s_default']: return None
    d_app = 1.0 - float(np.dot(a['emb'], b['emb']))
    terms, weights = [d_app], [cfg['w_app']]
    exact = False
    if a.get('plate') and b.get('plate') and a.get('plate_conf', 0) >= 0.5 and b.get('plate_conf', 0) >= 0.5:
        s = text_sim(a['plate'], b['plate']); terms.append(1.0 - s); weights.append(cfg['w_plate']); exact = s >= 0.99
    if a.get('coarse') and b.get('coarse'):
        if a['coarse'] != b['coarse'] and min(a.get('cls_conf', 0), b.get('cls_conf', 0)) >= 0.6: d_c = 1.0
        elif a.get('fine') != b.get('fine') and min(a.get('cls_conf', 0), b.get('cls_conf', 0)) >= 0.7: d_c = 0.6
        elif a.get('fine') == b.get('fine'): d_c = 0.0
        else: d_c = 0.3
        terms.append(d_c); weights.append(cfg['w_cls'])
    cost = sum(w * t for w, t in zip(weights, terms)) / sum(weights)
    if exact: cost -= cfg['plate_exact_bonus']
    return float(cost)

def associate_tracklets(trk, cfg, topology=None):
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import squareform
    topology = topology or {}; n = len(trk)
    if n == 0: return []
    if n == 1: return [0]
    D = np.full((n, n), 10.0, np.float64); np.fill_diagonal(D, 0.0)
    for i in range(n):
        for j in range(i + 1, n):
            c = tracklet_pair_cost(trk[i], trk[j], cfg, topology)
            if c is not None: D[i, j] = D[j, i] = max(c, 0.0)
    Z = linkage(squareform(D, checks=False), method='average')
    lab = fcluster(Z, t=cfg['assoc_thr'], criterion='distance')
    # إصلاح: لا يبقى تداخل زمني لنفس الكاميرا داخل هوية واحدة
    out = np.zeros(n, int); nxt = 0
    for cl in np.unique(lab):
        members = sorted(np.nonzero(lab == cl)[0], key=lambda k: trk[k]['t0']); groups = []
        for k in members:
            for g in groups:
                if all(trk[m]['cam'] != trk[k]['cam'] or trk[m]['t1'] < trk[k]['t0'] or trk[k]['t1'] < trk[m]['t0'] for m in g): g.append(k); break
            else: groups.append([k])
        for g in groups:
            for k in g: out[k] = nxt
            nxt += 1
    return out.tolist()

def quality_weighted_mean(embs, quals):
    e = np.asarray(embs, np.float32); q = np.asarray(quals, np.float32).clip(1e-3)
    m = (e * q[:, None]).sum(0) / q.sum(); return m / (np.linalg.norm(m) + 1e-9)
"""
exec(COMMON_SRC)
(RUN_DIR / 'common_logic_snapshot.py').write_text(COMMON_SRC, encoding='utf-8')
status('ok', 'المنطق المشترك جاهز')

✅ المنطق المشترك جاهز


## ٩. ReID العملاق: دمج الهياكل + رأس BNNeck + خسائر ID/Triplet/Circle
**المصادر:** VeRi (إن وُجدت) + مركباتك اليمنية في `yemen_reid_ids`. **الاختبار الذي يهمّك** هو هويات يمنية لم تُرَ في التدريب (تقسيم بالهوية، لا بالصورة).
**القياس:** mAP وCMC مع/بدون إعادة الترتيب، ومقارنة مع **خط الأساس** (البصمة العملاقة الخام بلا تدريب) لتعرف ما أضافه تدريبك فعلاً.

In [9]:
# ٩-أ. جدول ReID: VeRi + هوياتك اليمنية (تقسيم بالهوية)
def find_sub(root, names):
    for n in names:
        for p in Path(root).rglob(n):
            if p.is_dir(): return p
    return None
rows = []
if veri_imgs:
    d_tr, d_q, d_g = (find_sub(VERI_ROOT, [n]) for n in ('image_train', 'image_query', 'image_test'))
    for d, sp in ((d_tr, 'veri_train'), (d_q, 'veri_query'), (d_g, 'veri_gallery')):
        for p in (list_images(d) if d else []):
            m = VERI_RE.search(p.name)
            if m: rows.append({'path': str(p), 'pid': f'veri_{int(m.group(1)):04d}', 'cam': f'veri_c{int(m.group(2))}', 'source': 'veri', 'split': sp})
    if not d_tr: status('warn', 'لم أجد image_train داخل VeRi: لن تُستعمل')
yem_ids = sorted(d for d in YEM_ID_DIR.iterdir() if d.is_dir()) if YEM_ID_DIR.exists() else []
rng = random.Random(SEED); rng.shuffle(yem_ids)
n_eval = max(1, int(len(yem_ids) * 0.3)) if len(yem_ids) >= 6 else 0
for k, d in enumerate(yem_ids):
    sp = 'yem_eval' if k < n_eval else 'yem_train'
    for p in list_images(d):
        m = YEM_RE.search(p.stem); rows.append({'path': str(p), 'pid': f'yem_{d.name}', 'cam': f'yem_c{m.group(1) if m else 0}', 'source': 'yemen', 'split': sp})
REID_DF = pd.DataFrame(rows)
_reid_train = REID_DF[REID_DF.split.isin(['veri_train', 'yem_train'])] if len(REID_DF) else pd.DataFrame()
HAVE_REID = (len(_reid_train) > 100 and _reid_train.pid.nunique() >= 2
             and _reid_train.groupby('pid').size().ge(2).sum() >= 2)
if HAVE_REID:
    # قسم تقييم اليمن: query = صورة واحدة لكل (هوية، كاميرا)، gallery = الباقي
    ye = REID_DF[REID_DF.split == 'yem_eval']
    multi = ye.groupby('pid').cam.nunique(); good = set(multi[multi >= 2].index)
    qidx = ye[ye.pid.isin(good)].groupby(['pid', 'cam']).head(1).index
    REID_DF.loc[qidx, 'split'] = 'yem_query'; REID_DF.loc[(REID_DF.split == 'yem_eval'), 'split'] = 'yem_gallery'
    print(REID_DF.groupby(['source', 'split']).agg(imgs=('path', 'count'), ids=('pid', 'nunique')))
    assert not (set(REID_DF[REID_DF.split.isin(['yem_train', 'veri_train'])].pid) & set(REID_DF[REID_DF.split.isin(['yem_query', 'yem_gallery', 'veri_query', 'veri_gallery'])].pid)), 'تسرب هويات!'
    REID_DF.to_csv(RUN_DIR / 'reid_table.csv', index=False)
    if not len(REID_DF[REID_DF.split == 'yem_query']): status('warn', 'لا توجد هويات يمنية كافية (≥6 هويات، كل منها بكاميرتين) للتقييم → NOT_EVALUATED لليمن')
else:
    status('pause', 'بيانات ReID غير كافية: يلزم >100 صورة تدريب، وهويتان على الأقل، وصورتان لكل هوية على الأقل. أضف VeRi كاملة أو yemen_reid_ids ثم أعد الخلية')

⏸️ بيانات ReID غير كافية: يلزم >100 صورة تدريب، وهويتان على الأقل، وصورتان لكل هوية على الأقل. أضف VeRi كاملة أو yemen_reid_ids ثم أعد الخلية


In [10]:
# ٩-ب. تدريب رأس ReID (BNNeck) بخسائر ID + Batch-hard Triplet + Circle على ميزات الهياكل العملاقة
class ReIDHead(nn.Module):
    def __init__(self, d_in, d_out, n_id):
        super().__init__()
        self.pre = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, 2048), nn.GELU(), nn.Dropout(0.2), nn.Linear(2048, d_out))
        self.bn = nn.BatchNorm1d(d_out); self.bn.bias.requires_grad_(False)
        self.cls = nn.Linear(d_out, n_id, bias=False) if n_id else None
    def embed(self, x): return F.normalize(self.bn(self.pre(x)), dim=1)
    def forward(self, x):
        f = self.pre(x); g = self.bn(f)
        return F.normalize(f, dim=1), g, (self.cls(g) if self.cls is not None else None)

def batch_hard_triplet(emb, y, margin):
    d = 1 - emb @ emb.T; same = y[:, None] == y[None, :]
    dp = d.masked_fill(~same, -1).max(1).values; dn = d.masked_fill(same, 9).min(1).values
    return F.relu(dp - dn + margin).mean()
def circle_loss(emb, y, m, gamma):
    s = emb @ emb.T; same = y[:, None] == y[None, :]; eye = torch.eye(len(y), dtype=torch.bool, device=emb.device)
    pos, neg = same & ~eye, ~same
    ap = F.relu(1 + m - s.detach()); an = F.relu(s.detach() + m)
    lp = (-gamma * ap * (s - (1 - m))).masked_fill(~pos, -1e4); ln = (gamma * an * (s - m)).masked_fill(~neg, -1e4)
    ok = pos.any(1) & neg.any(1)
    return F.softplus(torch.logsumexp(lp, 1) + torch.logsumexp(ln, 1))[ok].mean() if ok.any() else s.sum() * 0

REID_ART = None
if HAVE_REID:
    tr = REID_DF[REID_DF.split.isin(['veri_train', 'yem_train'])].reset_index(drop=True)
    pid2i = {p: i for i, p in enumerate(sorted(tr.pid.unique()))}; ytr = np.array([pid2i[p] for p in tr.pid])
    Xtr = torch.from_numpy(build_feature_cache(list(tr.path), CFG['reid_views_per_image'], 'reid_train'))
    by_pid = collections.defaultdict(list)
    for i, y in enumerate(ytr): by_pid[y].append(i)
    ids = list(by_pid); P, Kk = CFG['reid_P'], CFG['reid_K']
    # الهويات اليمنية تُضاعف احتمال سحبها (بياناتك أهم من VeRi)
    is_yem = np.array([tr.source[by_pid[i][0]] == 'yemen' for i in ids]); wts = np.where(is_yem, 3.0, 1.0); wts = wts / wts.sum()
    reid = ReIDHead(ENC.dim, CFG['reid_embed_dim'], len(pid2i)).to(DEVICE)
    steps = max(20, math.ceil(len(ids) / P)); E = CFG['reid_epochs']
    opt = torch.optim.AdamW([p for p in reid.parameters() if p.requires_grad], lr=CFG['reid_lr'], weight_decay=CFG['reid_wd'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['reid_lr'], total_steps=E * steps)
    t0 = time.time()
    for ep in range(E):
        reid.train(); tot = 0
        for _ in range(steps):
            chosen = np.random.choice(len(ids), size=min(P, len(ids)), replace=False, p=wts); idx = []
            for c in chosen:
                pool = by_pid[ids[c]]; idx += list(np.random.choice(pool, Kk, replace=len(pool) < Kk))
            idx = np.array(idx); v = np.random.randint(0, Xtr.shape[1], len(idx))
            x = Xtr[idx, v].to(DEVICE).float(); y = torch.from_numpy(ytr[idx]).to(DEVICE)
            emb, g, logit = reid(x)
            loss = F.cross_entropy(logit, y, label_smoothing=CFG['label_smooth']) + batch_hard_triplet(emb, y, CFG['triplet_margin']) + 0.5 * circle_loss(F.normalize(g, dim=1), y, CFG['circle_m'], CFG['circle_gamma'])
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(reid.parameters(), 5.0); opt.step(); sched.step(); tot += float(loss)
        if ep % 5 == 0 or ep == E - 1: print(f'epoch {ep+1:3d}/{E} loss={tot/steps:.4f} ({time.time()-t0:.0f}s)')
        if (ep + 1) % 5 == 0 or ep == E - 1:
            torch.save({'epoch': ep + 1, 'state': {k: v.cpu() for k, v in reid.state_dict().items()}, 'optimizer': opt.state_dict(), 'loss': tot / steps}, RUN_DIR / 'reid_checkpoint.pt')
    reid.eval(); status('ok', 'انتهى تدريب ReID')

In [11]:
# ٩-ج. تقييم ReID (mAP / CMC) + مقارنة بخط الأساس + إعادة الترتيب — على هويات لم تُرَ في التدريب
def eval_reid(dist, qp, qc, gp, gc):
    aps, cmc = [], np.zeros(dist.shape[1]); n_ok = 0
    for i in range(dist.shape[0]):
        order = np.argsort(dist[i]); keep = ~((gp[order] == qp[i]) & (gc[order] == qc[i])); o = order[keep]
        match = (gp[o] == qp[i]).astype(np.int32)
        if match.sum() == 0: continue
        n_ok += 1; hit = np.cumsum(match); prec = hit / (np.arange(len(match)) + 1)
        aps.append((prec * match).sum() / match.sum()); first = np.argmax(match); cmc[first:] += 1
    return {'mAP': float(np.mean(aps)) if aps else float('nan'), 'rank1': float(cmc[0] / max(n_ok, 1)), 'rank5': float(cmc[min(4, len(cmc)-1)] / max(n_ok, 1)),
            'rank10': float(cmc[min(9, len(cmc)-1)] / max(n_ok, 1)), 'n_queries': n_ok}
def boot_ci_map(dist, qp, qc, gp, gc, B=200):
    rng_ = np.random.default_rng(SEED); vals = []
    for _ in range(B):
        sel = rng_.integers(0, dist.shape[0], dist.shape[0]); vals.append(eval_reid(dist[sel], qp[sel], qc[sel], gp, gc)['mAP'])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

@torch.no_grad()
def embed_with_head(X):
    out = []
    for i in range(0, len(X), 1024): out.append(reid.embed(torch.as_tensor(X[i:i+1024], device=DEVICE).float()).cpu().numpy())
    return np.concatenate(out)

REID_REPORT = {}
if HAVE_REID:
    for qs, gs, tag in (('yem_query', 'yem_gallery', 'yemen'), ('veri_query', 'veri_gallery', 'veri')):
        q, g = REID_DF[REID_DF.split == qs].reset_index(drop=True), REID_DF[REID_DF.split == gs].reset_index(drop=True)
        if len(q) < 5 or len(g) < 5: REID_REPORT[tag] = 'NOT_EVALUATED'; status('warn', f'{tag}: NOT_EVALUATED (بيانات غير كافية)'); continue
        Fq = build_feature_cache(list(q.path), 1, f'reid_{qs}')[:, 0].astype(np.float32); Fg = build_feature_cache(list(g.path), 1, f'reid_{gs}')[:, 0].astype(np.float32)
        qp, qc, gp, gc = q.pid.values, q.cam.values, g.pid.values, g.cam.values
        base_q = Fq / np.linalg.norm(Fq, axis=1, keepdims=True); base_g = Fg / np.linalg.norm(Fg, axis=1, keepdims=True)
        eq, eg = embed_with_head(Fq), embed_with_head(Fg)
        d_base, d_head = 1 - base_q @ base_g.T, 1 - eq @ eg.T
        r = {'baseline_raw_giant': eval_reid(d_base, qp, qc, gp, gc), 'trained_head': eval_reid(d_head, qp, qc, gp, gc)}
        r['trained_head']['mAP_ci95'] = boot_ci_map(d_head, qp, qc, gp, gc)
        if len(g) <= 8000:
            rk = CFG['rerank']; d_rr = re_ranking(d_head, 1 - eq @ eq.T, 1 - eg @ eg.T, rk['k1'], rk['k2'], rk['lambda'])
            r['trained_head+rerank'] = eval_reid(d_rr, qp, qc, gp, gc)
        else: r['trained_head+rerank'] = 'SKIPPED_gallery_too_large'
        REID_REPORT[tag] = r; print(tag, json.dumps(r, ensure_ascii=False, indent=1, default=_jd))
    save_json(RUN_DIR / 'reid_report.json', REID_REPORT)
    REID_ART = {'state': {k: v.cpu() for k, v in reid.state_dict().items()}, 'in_dim': ENC.dim, 'out_dim': CFG['reid_embed_dim'], 'n_train_ids': len(pid2i), 'report_path': str(RUN_DIR / 'reid_report.json')}

> **كيف تقرأ نتيجة ReID:** إن لم يتفوق `trained_head` على `baseline_raw_giant` على هوياتك اليمنية، فهذا يعني أن بيانات التدريب لا تضيف شيئاً — أضف هويات يمنية أكثر (من كاميرات متعددة وزوايا مختلفة) بدل زيادة حجم النموذج. وإن كان الفاصل `mAP_ci95` عريضاً فعدد الاستعلامات قليل ولا يصح الجزم بالتحسن.

## ١٠. اللوحات: كشف ← قراءة ← تفاصيل
1. **كاشف اللوحات** (YOLO11m) يُدرَّب على `plates/yolo` إن وُجدت، وإلا يُستخدم `production/plate_best.pt` إن وُجد.
2. **القارئ:** CRNN-CTC تدرّبه على لوحاتك (الأدق عادةً على خطوطك وتنسيقاتك) مقابل EasyOCR الجاهز؛ يُختار الأفضل على التحقق (CER).
3. **التفاصيل:** الرقم، المحافظة (بمطابقة تقريبية)، **فئة اللوحة من اللون** (تُتعلَّم من عيّناتك في `plates/categories/<الفئة>/`، ولا أفترض لك ألواناً).
4. **تصويت عبر الإطارات** يرفع الدقة بشكل ملحوظ لأن الإطار الواحد يخطئ كثيراً.

In [12]:
# ١٠-أ. تدريب كاشف اللوحات (إن توفرت بيانات YOLO) أو استعمال كاشف موجود
PLATE_DET_PATH = None
if INV['plate_yolo_yaml']:
    from ultralytics import YOLO
    try:
        m = YOLO(CFG['plate_det_weights'])
        m.train(data=str(PLATE_DIR / 'yolo' / 'data.yaml'), epochs=CFG['plate_det_epochs'], imgsz=CFG['plate_det_imgsz'], batch=16, patience=25,
                project=str(RUN_DIR / 'plate_det'), name='run', exist_ok=True, seed=SEED, device=0 if DEVICE == 'cuda' else 'cpu', plots=True)
        best = RUN_DIR / 'plate_det' / 'run' / 'weights' / 'best.pt'
        if best.is_file():
            PLATE_DET_PATH = best; v = YOLO(str(best)).val(data=str(PLATE_DIR / 'yolo' / 'data.yaml'), imgsz=CFG['plate_det_imgsz'], split='val', verbose=False)
            save_json(RUN_DIR / 'plate_detector_val.json', {'map50': float(v.box.map50), 'map50_95': float(v.box.map), 'precision': float(v.box.mp), 'recall': float(v.box.mr)})
            status('ok', f'كاشف اللوحات: mAP50={v.box.map50:.3f} mAP50-95={v.box.map:.3f} (على val الخاص بك)')
    except Exception as e:
        status('error', f'فشل تدريب كاشف اللوحات: {str(e)[:250]}')
if PLATE_DET_PATH is None:
    for cand in (PROD_DIR / 'plate_best.pt', DRIVE_ROOT / 'plate_best.pt'):
        if cand.is_file(): PLATE_DET_PATH = cand; status('ok', f'استعمال كاشف لوحات موجود: {cand}'); break
if PLATE_DET_PATH is None: status('pause', 'لا يوجد كاشف لوحات: ضع plates/yolo/data.yaml (صور+labels) أو production/plate_best.pt. يمكن مع ذلك قراءة قصاصات لوحات جاهزة.')

⏸️ لا يوجد كاشف لوحات: ضع plates/yolo/data.yaml (صور+labels) أو production/plate_best.pt. يمكن مع ذلك قراءة قصاصات لوحات جاهزة.


In [13]:
# ١٠-ب. فئة اللوحة من اللون: تُتعلَّم من عيّناتك plates/categories/<الفئة>/*.jpg (بلا افتراضات جاهزة)
def plate_color_feat(im):
    a = np.asarray(im.convert('RGB').resize((96, 24), Image.BILINEAR), np.float32)
    import cv2
    lab = cv2.cvtColor(a.astype(np.uint8), cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
    km = cv2.kmeans(lab, 2, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 15, 0.5), 2, cv2.KMEANS_PP_CENTERS)
    labels, centers = km[1].ravel(), km[2]; bg = centers[np.bincount(labels).argmax()]   # الخلفية = العنقود الأكبر
    return bg.tolist()
PLATE_COLOR = None
cat_dir = PLATE_DIR / 'categories'
if cat_dir.exists():
    X, Y = [], []
    for d in sorted(x for x in cat_dir.iterdir() if x.is_dir()):
        for p in list_images(d): X.append(plate_color_feat(_load(p))); Y.append(d.name)
    if len(set(Y)) >= 2 and len(Y) >= 10:
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.model_selection import cross_val_score
        knn = KNeighborsClassifier(n_neighbors=min(5, max(1, len(Y)//len(set(Y))))).fit(X, Y)
        min_class_n = min(collections.Counter(Y).values())
        n_splits = min(5, min_class_n)
        cv = cross_val_score(KNeighborsClassifier(n_neighbors=min(3, max(1, len(Y) - math.ceil(len(Y) / n_splits)))), X, Y, cv=n_splits, scoring='accuracy') if min_class_n >= 2 else None
        PLATE_COLOR = {'X': X, 'Y': Y, 'k': knn.n_neighbors}
        status('ok', f'نموذج لون اللوحة: فئات={sorted(set(Y))} | دقة تحقق متقاطع={None if cv is None else round(float(cv.mean()), 3)}')
        save_json(RUN_DIR / 'plate_color_model.json', PLATE_COLOR)
    else: status('warn', 'عيّنات الفئات قليلة (≥2 فئة و≥10 صور) → فئة اللوحة ستكون unknown')
else: status('pause', 'لا يوجد plates/categories: فئة اللوحة (خاص/أجرة/حكومي…) ستكون unknown حتى تضع عيّنات مسمّاة')

⏸️ لا يوجد plates/categories: فئة اللوحة (خاص/أجرة/حكومي…) ستكون unknown حتى تضع عيّنات مسمّاة


In [14]:
# ١٠-ج. قارئ CRNN-CTC على لوحاتك + مقارنة مع EasyOCR — التقسيم بمجموعات نص اللوحة (لا تسرب)
PLATE_OCR = None
lab_csv = PLATE_DIR / 'labels.csv'     # أعمدة: file,text  (file نسبي إلى plates/images)
if lab_csv.is_file():
    df = pd.read_csv(lab_csv, dtype=str).dropna(); df['norm'] = df.text.map(normalize_ar)
    df['path'] = df.file.map(lambda f: str(PLATE_DIR / 'images' / f)); df = df[df.path.map(os.path.isfile)].reset_index(drop=True)
    texts = sorted(df.norm.unique()); random.Random(SEED).shuffle(texts)
    nt = len(texts)
    n_test = max(1, int(round(nt * 0.15))) if nt >= 7 else 0
    n_val = max(1, int(round(nt * 0.15))) if nt >= 5 else 0
    te, va = set(texts[:n_test]), set(texts[n_test:n_test+n_val])
    df['split'] = df.norm.map(lambda t: 'test' if t in te else 'val' if t in va else 'train')
    print(df.split.value_counts().to_dict())
    charset = ['<blank>'] + sorted(set(''.join(df.norm))); c2i = {c: i for i, c in enumerate(charset)}
    H, W = CFG['plate_img_h'], CFG['plate_img_w']
    def prep(im, train):
        im = im.convert('RGB')
        if train:
            im = T.RandomAffine(4, translate=(0.03, 0.06), scale=(0.9, 1.1), shear=6)(im); im = T.ColorJitter(0.5, 0.5, 0.4, 0.03)(im)
            if random.random() < 0.3: im = _lowres(im)
            if random.random() < 0.2: im = im.filter(__import__('PIL.ImageFilter', fromlist=['x']).GaussianBlur(random.uniform(0.3, 1.5)))
        return torch.from_numpy(np.asarray(im.resize((W, H), Image.BICUBIC), np.float32) / 255.).permute(2, 0, 1)
    class CRNN(nn.Module):
        def __init__(self, nc):
            super().__init__()
            def blk(i, o, pool): return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), *( [nn.MaxPool2d(pool)] if pool else []))
            self.cnn = nn.Sequential(blk(3, 64, (2, 2)), blk(64, 128, (2, 2)), blk(128, 256, None), blk(256, 256, (2, 1)), blk(256, 384, None), blk(384, 384, (2, 1)))
            self.rnn = nn.LSTM(384, 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2); self.fc = nn.Linear(512, nc)
        def forward(self, x):
            f = self.cnn(x); f = f.mean(2).permute(0, 2, 1); o, _ = self.rnn(f); return self.fc(o)    # (B,T,C)
    def ctc_decode(logits):
        p = logits.softmax(-1); conf, idx = p.max(-1); res = []
        for b in range(idx.shape[0]):
            s, cs, prev = [], [], 0
            for t in range(idx.shape[1]):
                k = int(idx[b, t])
                if k != 0 and k != prev: s.append(charset[k]); cs.append(float(conf[b, t]))
                prev = k
            res.append((''.join(s), float(np.mean(cs)) if cs else 0.0))
        return res
    parts_p = {s: df[df.split == s].reset_index(drop=True) for s in ('train', 'val', 'test')}
    if len(parts_p['train']) >= 50 and len(parts_p['val']) >= 10:
        net = CRNN(len(charset)).to(DEVICE); ctc = nn.CTCLoss(blank=0, zero_infinity=True)
        opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4); BS = 64
        E = CFG['plate_ocr_epochs']; sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=E * math.ceil(len(parts_p['train']) / BS))
        imgs_cache = {s: [_load(p) for p in parts_p[s].path] for s in parts_p}
        def run_eval(split):
            net.eval(); out = []
            with torch.no_grad():
                for i in range(0, len(imgs_cache[split]), 128):
                    x = torch.stack([prep(im, False) for im in imgs_cache[split][i:i+128]]).to(DEVICE); out += ctc_decode(net(x).float().cpu())
            gt = list(parts_p[split].norm); em = float(np.mean([a == b[0] for a, b in zip(gt, out)])) if gt else float('nan')
            cer = float(np.sum([levenshtein(a, b[0]) for a, b in zip(gt, out)]) / max(1, sum(len(a) for a in gt)))
            return em, cer, out
        best, best_state, bad = 9, None, 0
        for ep in range(E):
            net.train(); perm = np.random.permutation(len(parts_p['train']))
            for i in range(0, len(perm), BS):
                ix = perm[i:i+BS]; x = torch.stack([prep(imgs_cache['train'][k], True) for k in ix]).to(DEVICE)
                tg = [torch.tensor([c2i[c] for c in parts_p['train'].norm[k]]) for k in ix]
                lp = net(x).log_softmax(-1).permute(1, 0, 2)
                loss = ctc(lp, torch.cat(tg).to(DEVICE), torch.full((len(ix),), lp.shape[0], dtype=torch.long), torch.tensor([len(t) for t in tg]))
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step(); sched.step()
            if ep % 4 == 3 or ep == E - 1:
                em, cer, _ = run_eval('val'); print(f'epoch {ep+1} val exact={em:.3f} CER={cer:.3f}')
                if cer < best: best, best_state, bad = cer, copy.deepcopy(net.state_dict()), 0
                else:
                    bad += 1
                    if bad >= 5: break
        if best_state: net.load_state_dict(best_state)
        res = {}
        if len(parts_p['test']) >= 5:
            em, cer, out = run_eval('test'); k = int(round(em * len(parts_p['test'])))
            res['crnn'] = {'exact_match': wilson(k, len(parts_p['test'])), 'CER': cer, 'n_test': len(parts_p['test'])}
            # EasyOCR كخط أساس على الاختبار نفسه
            try:
                import easyocr; rd = easyocr.Reader(CFG['plate_langs'], gpu=(DEVICE == 'cuda'), verbose=False)
                eo = [normalize_ar(' '.join(rd.readtext(np.asarray(im), detail=0))) for im in imgs_cache['test']]; gt = list(parts_p['test'].norm)
                res['easyocr'] = {'exact_match': wilson(sum(a == b for a, b in zip(gt, eo)), len(gt)), 'CER': float(sum(levenshtein(a, b) for a, b in zip(gt, eo)) / max(1, sum(len(a) for a in gt)))}
            except Exception as e: res['easyocr'] = f'NOT_EVALUATED: {str(e)[:120]}'
        else: res['note'] = 'NOT_EVALUATED: اختبار OCR أقل من 5 لوحات (بنص فريد). زد البيانات.'
        print(json.dumps(res, ensure_ascii=False, indent=1, default=_jd)); save_json(RUN_DIR / 'plate_ocr_report.json', res)
        PLATE_OCR = {'state': {k: v.cpu() for k, v in net.state_dict().items()}, 'charset': charset, 'H': H, 'W': W, 'report': res}
        net.eval()
    else: status('warn', 'لوحات التدريب <50 أو التحقق <10 → لا تدريب OCR. استعمل EasyOCR فقط أو زد البيانات.')
else: status('pause', 'لا يوجد plates/labels.csv (file,text) → لا تدريب OCR. سيعمل EasyOCR فقط في التشغيل.')

⏸️ لا يوجد plates/labels.csv (file,text) → لا تدريب OCR. سيعمل EasyOCR فقط في التشغيل.


In [15]:
# ١٠-د. خط معالجة اللوحة الكامل لقصاصة مركبة: كشف ← قراءة (CRNN أو EasyOCR) ← تحليل ← فئة اللون
_EASY = {'r': None}
def _easy():
    if _EASY['r'] is None:
        import easyocr; _EASY['r'] = easyocr.Reader(CFG['plate_langs'], gpu=(DEVICE == 'cuda'), verbose=False)
    return _EASY['r']
_PD = {'m': None}
def detect_plates(img_pil, conf=0.25):
    if PLATE_DET_PATH is None: return []
    if _PD['m'] is None:
        from ultralytics import YOLO; _PD['m'] = YOLO(str(PLATE_DET_PATH))
    r = _PD['m'].predict(np.asarray(img_pil.convert('RGB')), conf=conf, imgsz=CFG['plate_det_imgsz'], verbose=False)[0]
    return [(tuple(map(int, b.xyxy[0].tolist())), float(b.conf[0])) for b in r.boxes]
def read_plate_crop(crop):
    if PLATE_OCR is not None:
        if 'net_eval' not in PLATE_OCR:
            n = CRNN(len(PLATE_OCR['charset'])).to(DEVICE); n.load_state_dict({k: v.to(DEVICE) for k, v in PLATE_OCR['state'].items()}); PLATE_OCR['net_eval'] = n.eval()
        with torch.no_grad(): t, c = ctc_decode(PLATE_OCR['net_eval'](prep(crop, False)[None].to(DEVICE)).float().cpu())[0]
        if t: return t, c
    try:
        rr = _easy().readtext(np.asarray(crop.convert('RGB')), detail=1)
        if rr: return ' '.join(x[1] for x in rr), float(np.mean([x[2] for x in rr]))
    except Exception: pass
    return '', 0.0
def plate_category(crop):
    if not PLATE_COLOR: return 'unknown'
    from sklearn.neighbors import KNeighborsClassifier
    k = KNeighborsClassifier(n_neighbors=PLATE_COLOR['k']).fit(PLATE_COLOR['X'], PLATE_COLOR['Y']); return str(k.predict([plate_color_feat(crop)])[0])
def analyze_vehicle_plates(vehicle_crops, max_frames=10):
    """vehicle_crops: قائمة صور PIL لنفس المركبة (عدة إطارات). يعيد قراءة مصوَّتة وتفاصيل."""
    reads, cats = [], []
    for im in sorted(vehicle_crops, key=lambda i: -i.size[0]*i.size[1])[:max_frames]:
        for (x1, y1, x2, y2), c in detect_plates(im)[:1]:
            pc = im.crop((x1, y1, x2, y2)); t, rc = read_plate_crop(pc)
            if t: reads.append((t, rc * c)); cats.append(plate_category(pc))
    v = vote_plate(reads)
    info = parse_plate(v['text'], YEMEN_GOVERNORATES_AR) if v['text'] else None
    cat = collections.Counter(cats).most_common(1)[0][0] if cats else 'unknown'
    return {'vote': v, 'parsed': info, 'plate_category': cat, 'n_reads': len(reads)}
status('ok', 'خط معالجة اللوحة جاهز')

✅ خط معالجة اللوحة جاهز


## ١١. التتبع متعدد الكاميرات (MTMC)
**المسار:** كاشفك + ByteTrack/BoT-SORT لكل كاميرا ← اختيار أفضل إطارات لكل مسار (حجم × ثقة × حدّة) ← بصمة ReID موزونة بالجودة + فئة مصوَّتة + لوحة مصوَّتة ← **ربط هرمي** بين كل المسارات بدمج المظهر والفئة واللوحة مع قيود زمنية/طوبولوجية (مركبتان متداخلتان زمنياً في كاميرا واحدة لا تُدمجان أبداً).
**ما تنقله اللوحة:** إن قُرئت اللوحة بثقة في كاميرتين وتطابقت، يُخفَّض الثمن بقوة (دليل أقوى من المظهر)؛ وإن اختلفت بثقة عالية يُرفع الثمن.

In [16]:
# ١١-أ. ضبط عتبة الربط على اختبار بديل (هوياتك اليمنية بكاميرات متعددة) — الضبط على نصف الهويات والحكم على النصف الآخر
def pairwise_f1(labels, pids, cams):
    tp = fp = fn = 0; n = len(labels)
    for i in range(n):
        for j in range(i + 1, n):
            if cams[i] == cams[j]: continue
            same_t, same_p = pids[i] == pids[j], labels[i] == labels[j]
            tp += same_t and same_p; fp += (not same_t) and same_p; fn += same_t and (not same_p)
    pr, rc = tp / max(1, tp + fp), tp / max(1, tp + fn); return (2 * pr * rc / max(1e-9, pr + rc)), pr, rc

def make_pseudo_tracklets(df, F_emb):
    trk, meta = [], []
    for (pid, cam), g in df.groupby(['pid', 'cam']):
        e = F_emb[g.index.values].mean(0); e /= np.linalg.norm(e) + 1e-9; k = len(trk) * 1e-3
        trk.append({'cam': cam, 't0': k, 't1': k, 'emb': e, 'coarse': None, 'fine': None}); meta.append((pid, cam))
    return trk, meta

ASSOC_TUNED = dict(CFG['mtmc'])
if HAVE_REID and (REID_DF.split.isin(['yem_query', 'yem_gallery'])).any():
    ye = REID_DF[REID_DF.split.isin(['yem_query', 'yem_gallery'])].reset_index(drop=True)
    Fye = build_feature_cache(list(ye.path), 1, 'reid_yem_all')[:, 0].astype(np.float32); Eye = embed_with_head(Fye)
    pids_all = sorted(ye.pid.unique()); random.Random(SEED).shuffle(pids_all); half = set(pids_all[:len(pids_all)//2])
    res = {}
    for name, sel in (('tune', ye.pid.isin(half)), ('hold', ~ye.pid.isin(half))):
        sub = ye[sel].reset_index(drop=True); trk, meta = make_pseudo_tracklets(sub, Eye[sel.values]); res[name] = (trk, [m[0] for m in meta], [m[1] for m in meta])
    best_thr, best_f = None, -1
    for thr in np.arange(0.10, 0.80, 0.04):
        c = dict(CFG['mtmc'], assoc_thr=float(thr), max_gap_s_default=1e9)
        lab = associate_tracklets(res['tune'][0], c, {}); f = pairwise_f1(lab, res['tune'][1], res['tune'][2])[0]
        if f > best_f: best_thr, best_f = float(thr), f
    c = dict(CFG['mtmc'], assoc_thr=best_thr, max_gap_s_default=1e9); lab = associate_tracklets(res['hold'][0], c, {})
    f, pr, rc = pairwise_f1(lab, res['hold'][1], res['hold'][2]); ASSOC_TUNED['assoc_thr'] = best_thr
    status('ok', f'عتبة الربط المضبوطة={best_thr:.2f} (F1 الضبط={best_f:.3f}) | على النصف المحجوب: pairF1={f:.3f} P={pr:.3f} R={rc:.3f}')
    save_json(RUN_DIR / 'assoc_tuning.json', {'thr': best_thr, 'tune_f1': best_f, 'holdout': {'f1': f, 'precision': pr, 'recall': rc}, 'note': 'اختبار بديل بلا زمن/لوحة؛ لا يقيس أخطاء التتبع'})
else: status('pause', 'لا هويات يمنية للتقييم → العتبة الافتراضية من CFG')

⏸️ لا هويات يمنية للتقييم → العتبة الافتراضية من CFG


In [17]:
# ١١-ب. استخراج المسارات من فيديو كاميرا: كاشف + تتبع + أفضل الإطارات لكل مسار
import cv2, heapq
EXCLUDE_NAMES = {'person', 'pedestrian', 'people', 'human', 'rider'}     # نتجاهل المشاة مؤقتاً كما طلبت (human hauler مركبة وتبقى)
MAX_CROPS = 24
def _detector_path():
    for p in (PROD_DIR / 'detector_best.pt', DRIVE_ROOT / 'detection_yolo11s_best.pt', DRIVE_ROOT / 'best.pt'):
        if p.is_file(): return p
    return None
DET_PATH = _detector_path()
if DET_PATH is None: status('warn', 'لا يوجد كاشف مركبات في production/: سيُنزَّل yolo11m.pt (COCO) كبديل')
def _quality(crop_bgr, conf):
    g = cv2.cvtColor(cv2.resize(crop_bgr, (64, 64)), cv2.COLOR_BGR2GRAY); sharp = min(1.0, cv2.Laplacian(g, cv2.CV_64F).var() / 120.0)
    h, w = crop_bgr.shape[:2]; return math.sqrt(h * w) * conf * (0.4 + 0.6 * sharp)

def extract_raw_tracklets(video_path, cam_id, clock_offset_s=0.0, max_frames=None):
    from ultralytics import YOLO
    model = YOLO(str(DET_PATH) if DET_PATH else 'yolo11m.pt'); cap = cv2.VideoCapture(str(video_path)); fps = cap.get(cv2.CAP_PROP_FPS) or 25.0; cap.release()
    tr = {}; fi = -1
    for r in model.track(source=str(video_path), stream=True, persist=True, tracker=CFG['mtmc']['tracker'], conf=CFG['mtmc']['det_conf'], imgsz=CFG['mtmc']['det_imgsz'], verbose=False, device=0 if DEVICE == 'cuda' else 'cpu'):
        fi += 1
        if max_frames and fi >= max_frames: break
        if r.boxes is None or r.boxes.id is None: continue
        img = r.orig_img
        for b, tid, c, k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(), r.boxes.conf.cpu().numpy(), r.boxes.cls.int().cpu().tolist()):
            if str(r.names.get(k, k)).lower() in EXCLUDE_NAMES: continue
            x1, y1, x2, y2 = [int(v) for v in b]; x1, y1 = max(0, x1), max(0, y1)
            if x2 - x1 < 32 or y2 - y1 < 32: continue
            t = tr.setdefault(tid, {'cam': cam_id, 'tid': int(tid), 'f0': fi, 'f1': fi, 'boxes': [], 'crops': [], 'det_cls': collections.Counter()})
            t['f1'] = fi; t['boxes'].append((fi, x1, y1, x2, y2)); t['det_cls'][str(r.names.get(k, k))] += 1
            if fi % 2 == 0:
                crop = img[y1:y2, x1:x2]
                if crop.size == 0: continue
                q = _quality(crop, float(c)); item = (q, fi, Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
                if len(t['crops']) < MAX_CROPS: heapq.heappush(t['crops'], item)
                elif q > t['crops'][0][0]: heapq.heapreplace(t['crops'], item)
    out = []
    for t in tr.values():
        if len(t['boxes']) < CFG['mtmc']['min_tracklet_len'] or not t['crops']: continue
        t['t0'], t['t1'], t['fps'] = clock_offset_s + t['f0'] / fps, clock_offset_s + t['f1'] / fps, fps; out.append(t)
    return out

def summarize_tracklet(t):
    pils = [c[2] for c in sorted(t['crops'], key=lambda c: -c[0])]; quals = [c[0] for c in sorted(t['crops'], key=lambda c: -c[0])]
    Fm = ENC.encode_many(pils)
    if HAVE_REID: E = embed_with_head(Fm)
    else: E = Fm / (np.linalg.norm(Fm, axis=1, keepdims=True) + 1e-9)
    s = {'cam': t['cam'], 'tid': t['tid'], 't0': t['t0'], 't1': t['t1'], 'n_frames': len(t['boxes']), 'emb': quality_weighted_mean(E, quals), 'det_cls': t['det_cls'].most_common(1)[0][0]}
    if VEH_ART is not None:
        with torch.no_grad(): p = F.softmax(head(torch.from_numpy(Fm).to(DEVICE).float()) / VEH_ART['temperature'], 1).mean(0).cpu().numpy()
        k = int(p.argmax()); ans = p[k] >= VEH_ART['reject_threshold']
        s.update({'fine': VEH_ART['classes'][k] if ans else 'unknown', 'coarse': VEH_ART['coarse_of'][VEH_ART['classes'][k]] if ans else None, 'cls_conf': float(p[k]),
                  'top3': [(VEH_ART['classes'][i], float(p[i])) for i in np.argsort(-p)[:3]]})
    if PLATE_DET_PATH is not None:
        pa = analyze_vehicle_plates(pils); s.update({'plate': pa['vote']['text'] or None, 'plate_conf': pa['vote']['conf'], 'plate_info': pa['parsed'], 'plate_category': pa['plate_category']})
    s['best_crop'] = pils[0]; return s
status('ok', 'دوال التتبع والتلخيص جاهزة')

✅ دوال التتبع والتلخيص جاهزة


In [18]:
# ١١-ج. تشغيل كل كاميرات mtmc_videos ← ربط عالمي ← نتائج (مع حماية خصوصية اللوحات)
CAM_CLOCK_OFFSETS = {}      # مثال: {'cam02': 12.5}  (ثوانٍ) لمزامنة ساعات الكاميرات إن لم تكن متزامنة
MAX_FRAMES_PER_VIDEO = None  # للتجربة السريعة ضع مثلاً 3000
GLOBAL = None
vids = sorted(VIDEO_DIR.glob('*.mp4')) if VIDEO_DIR.exists() else []
if not vids: status('pause', 'لا فيديوهات في mtmc_videos/ — ضع <cam_id>.mp4 ثم أعد الخلية')
else:
    summaries = []
    for v in vids:
        cam = v.stem; t0 = time.time(); raw = extract_raw_tracklets(v, cam, CAM_CLOCK_OFFSETS.get(cam, 0.0), MAX_FRAMES_PER_VIDEO)
        sm = [summarize_tracklet(t) for t in tqdm(raw, desc=f'تلخيص {cam}')]
        for s, t in zip(sm, raw): s['boxes'] = t['boxes']
        summaries += sm; status('ok', f'{cam}: {len(sm)} مسار ({time.time()-t0:.0f}s)')
    cfg_run = dict(ASSOC_TUNED); topo = {k: tuple(v) for k, v in CAMERA_TOPOLOGY.items()}
    labels = associate_tracklets(summaries, cfg_run, topo)
    P = CFG['PRIVACY']; rows = []; gal = RUN_DIR / 'global_gallery'; gal.mkdir(exist_ok=True)
    for s, g in zip(summaries, labels):
        pl = s.get('plate'); rows.append({'global_id': g, 'cam': s['cam'], 'track_id': s['tid'], 't0': round(s['t0'], 2), 't1': round(s['t1'], 2), 'frames': s['n_frames'],
            'vehicle_class': s.get('fine'), 'coarse': s.get('coarse'), 'class_conf': s.get('cls_conf'), 'detector_class': s['det_cls'],
            'plate_hash': hash_plate(pl, P['salt']) if (pl and P['hash_plates']) else None, 'plate_text': pl if (pl and P['store_raw_plate']) else None,
            'plate_conf': s.get('plate_conf'), 'governorate': (s.get('plate_info') or {}).get('governorate'), 'plate_category': s.get('plate_category')})
        s['best_crop'].save(gal / f'g{g:05d}_{s["cam"]}_{s["tid"]}.jpg')
    GLOBAL = pd.DataFrame(rows).sort_values(['global_id', 't0']); GLOBAL.to_csv(RUN_DIR / 'global_tracks.csv', index=False, encoding='utf-8-sig')
    multi = GLOBAL.groupby('global_id').cam.nunique(); status('ok', f'{len(summaries)} مسار → {GLOBAL.global_id.nunique()} هوية عالمية | هويات ظهرت في ≥2 كاميرا: {(multi >= 2).sum()}')
    display(GLOBAL.head(20))

⏸️ لا فيديوهات في mtmc_videos/ — ضع <cam_id>.mp4 ثم أعد الخلية


In [19]:
# ١١-د. (اختياري) IDF1 الحقيقي إن وفّرت حقيقة أرضية: mtmc_gt/<cam>.txt بصيغة MOT (frame,id,x,y,w,h,...) وبمعرّفات عالمية موحّدة بين الكاميرات
GT_DIR = DATA_ROOT / 'mtmc_gt'
if GLOBAL is not None and GT_DIR.exists() and list(GT_DIR.glob('*.txt')):
    import motmetrics as mm
    acc = mm.MOTAccumulator(auto_id=False)
    hyp = collections.defaultdict(lambda: collections.defaultdict(list))   # cam -> frame -> [(gid,x,y,w,h)]
    for s, g in zip(summaries, labels):
        for (f, x1, y1, x2, y2) in s['boxes']: hyp[s['cam']][f].append((g, x1, y1, x2 - x1, y2 - y1))
    for ci, gtf in enumerate(sorted(GT_DIR.glob('*.txt'))):
        cam = gtf.stem; gt = collections.defaultdict(list)
        for ln in gtf.read_text().splitlines():
            a = ln.split(',')
            if len(a) >= 6: gt[int(float(a[0])) - 1].append((int(float(a[1])), *map(float, a[2:6])))
        for f in sorted(set(gt) | set(hyp[cam])):
            gi = [x[0] for x in gt.get(f, [])]; hi = [x[0] for x in hyp[cam].get(f, [])]
            d = mm.distances.iou_matrix(np.array([x[1:] for x in gt.get(f, [])]).reshape(-1, 4), np.array([x[1:] for x in hyp[cam].get(f, [])]).reshape(-1, 4), max_iou=0.5) if gi and hi else np.empty((len(gi), len(hi)))
            acc.update(gi, hi, d, frameid=ci * 10**7 + f)
    summ = mm.metrics.create().compute(acc, metrics=['idf1', 'idp', 'idr', 'num_switches', 'mota', 'num_fragmentations'], name='MTMC')
    print(summ.T); save_json(RUN_DIR / 'mtmc_idf1.json', summ.iloc[0].to_dict())
else: status('pause', 'لا حقيقة أرضية MTMC (mtmc_gt/) → IDF1 NOT_EVALUATED. لا تثق بدقة الربط حتى تقيسها على بياناتك')

⏸️ لا حقيقة أرضية MTMC (mtmc_gt/) → IDF1 NOT_EVALUATED. لا تثق بدقة الربط حتى تقيسها على بياناتك


## ١٢. اختبار دخان للمنطق الخالص (بلا GPU)
يتحقق من أن الربط لا يدمج مركبتين متداخلتين في كاميرا واحدة، وأن اللوحة المتطابقة تجمع، وأن محلّل اللوحة والتصويت يعملان. **اجتياز هذا الاختبار لا يقيس دقة النماذج**، بل سلامة المنطق فقط.

In [20]:
# ١٢. اختبار دخان: بيانات اصطناعية صغيرة
rs = np.random.default_rng(1)
def _unit(v): return v / np.linalg.norm(v)
base = [_unit(rs.normal(size=64)) for _ in range(3)]
def mk(cam, t0, t1, v, plate=None, fine='taxi', coarse='car'):
    return {'cam': cam, 't0': t0, 't1': t1, 'emb': _unit(base[v] + 0.04 * rs.normal(size=64)), 'fine': fine, 'coarse': coarse, 'cls_conf': 0.9, 'plate': plate, 'plate_conf': 0.9 if plate else 0}
trk = [mk('c1', 0, 10, 0, '12345 تعز'), mk('c2', 30, 40, 0, '12345 تعز'), mk('c1', 5, 15, 1, '77777 صنعاء', 'hilux', 'pickup'),
       mk('c2', 50, 60, 1, '77777 صنعاء', 'hilux', 'pickup'), mk('c1', 100, 110, 2, None, 'bus_daewoo', 'bus'), mk('c2', 140, 150, 2, None, 'bus_daewoo', 'bus')]
lab = associate_tracklets(trk, dict(CFG['mtmc']), {})
assert lab[0] == lab[1] and lab[2] == lab[3] and lab[4] == lab[5], f'ربط خاطئ: {lab}'
assert len({lab[0], lab[2], lab[4]}) == 3, f'دمج مركبات مختلفة: {lab}'
# تداخل زمني في كاميرا واحدة ممنوع حتى لو تطابق المظهر
a, b = mk('c1', 0, 10, 0), mk('c1', 5, 12, 0); assert associate_tracklets([a, b], dict(CFG['mtmc']), {})[0] != associate_tracklets([a, b], dict(CFG['mtmc']), {})[1]
# قيد الطوبولوجيا: خارج النافذة = لا ربط
c = mk('c1', 0, 10, 0), mk('c2', 400, 410, 0); assert associate_tracklets(list(c), dict(CFG['mtmc']), {('c1', 'c2'): (5, 120)})[0] != associate_tracklets(list(c), dict(CFG['mtmc']), {('c1', 'c2'): (5, 120)})[1]
# اللوحة
pp = parse_plate('٣٢٤١٥ تعز', YEMEN_GOVERNORATES_AR); assert pp['number'] == '32415' and pp['governorate'] == 'تعز', pp
v = vote_plate([('12345 تعز', .9), ('12845 تعز', .4), ('12345 تعز', .7)]); assert v['text'] == '12345 تعز', v
# إعادة الترتيب: الشكل صحيح ولا ينهار
Eq = np.stack([_unit(rs.normal(size=16)) for _ in range(5)]); Eg = np.stack([_unit(rs.normal(size=16)) for _ in range(30)])
rr = re_ranking(1 - Eq @ Eg.T, 1 - Eq @ Eq.T, 1 - Eg @ Eg.T, 5, 3, 0.3); assert rr.shape == (5, 30) and np.isfinite(rr).all()
status('ok', 'اختبارات الدخان نجحت (منطق الربط/اللوحة/إعادة الترتيب سليم)')

✅ اختبارات الدخان نجحت (منطق الربط/اللوحة/إعادة الترتيب سليم)


## ١٣. التصدير النهائي إلى Drive — في مكان منفصل
يُكتب كل شيء في `YemenVision_MTMC/final_models/stage9_<RUN_ID>/`، ويُنسخ إلى `final_models/latest/` **فقط إذا اجتازت البوابات**. الرؤوس المدرّبة صغيرة (ميغابايتات)؛ أما الهياكل العملاقة نفسها فتُحمَّل من `model_zoo/` (أو من الإنترنت) عند التشغيل، ولا تُكرَّر هنا إلا إن فعّلت `SAVE_BACKBONE_WEIGHTS` (قد تبلغ عشرات الغيغابايتات).
**بوابات الترقية:** ReID: الحد الأدنى لفاصل mAP على اليمن ≥ خط الأساس الخام | الفئات: الحد الأدنى لدقة Top-1 ≥ العتبة | OCR: الحد الأدنى للمطابقة التامة ≥ العتبة. ما لم يُقَس لا يُحسب نجاحاً ولا فشلاً (يُعلَن `NOT_EVALUATED`).

In [21]:
# ١٣-أ. وحدة التشغيل النهائية المستقلة (تُكتب إلى Drive مع المنطق المشترك نفسه)
RUNTIME_HEAD = r"""
# YemenVision Stage 9 runtime — مولَّد تلقائياً من الدفتر. الاستخدام:
#   from yemenvision_stage9_runtime import YemenVisionStage9
#   yv = YemenVisionStage9('/content/drive/MyDrive/YemenVision_MTMC/final_models/latest')
"""
RUNTIME_BODY = r"""
import os, json, time, collections
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image

CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
IN_MEAN, IN_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class _One(nn.Module):
    def __init__(self, spec, device, fp16):
        super().__init__()
        self.spec, self.device = spec, device
        self.dtype = torch.float16 if (fp16 and device == 'cuda') else torch.float32
        k, mid = spec['kind'], spec['id']
        if k == 'dinov2':
            self.m = torch.hub.load('facebookresearch/dinov2', mid); self.mean, self.std = IN_MEAN, IN_STD
        elif k == 'openclip':
            import open_clip
            name, tag = mid.split('|'); self.m, _, _ = open_clip.create_model_and_transforms(name, pretrained=tag); self.mean, self.std = CLIP_MEAN, CLIP_STD
        else:
            import timm
            self.m = timm.create_model(mid, pretrained=True, num_classes=0)
            c = timm.data.resolve_data_config({}, model=self.m); self.mean, self.std = c['mean'], c['std']; spec['res'] = int(c['input_size'][-1])
        self.m = self.m.to(device).to(self.dtype).eval()
        for p in self.m.parameters(): p.requires_grad_(False)
        self.res = int(spec['res'])
        self.register_buffer('mu', torch.tensor(self.mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('sd', torch.tensor(self.std).view(1, 3, 1, 1), persistent=False)
        self.to(device)
    def forward(self, x):
        x = ((x - self.mu) / self.sd).to(self.dtype); k = self.spec['kind']
        if k == 'dinov2':
            o = self.m.forward_features(x); f = torch.cat([o['x_norm_clstoken'], o['x_norm_patchtokens'].mean(1)], dim=1)
        elif k == 'openclip': f = self.m.encode_image(x)
        else: f = self.m(x)
        return F.normalize(f.float(), dim=1)

class GiantEncoder:
    def __init__(self, specs, device, fp16=True, bs=64):
        self.parts = [_One(dict(s), device, fp16) for s in specs]; self.device, self.bs = device, bs
    @torch.no_grad()
    def encode_pils(self, pils):
        outs = []
        for p in self.parts:
            arr = np.stack([np.asarray(im.convert('RGB').resize((p.res, p.res), Image.BICUBIC)) for im in pils])
            x = torch.from_numpy(arr).to(self.device).permute(0, 3, 1, 2).float() / 255.0; outs.append(p(x))
        return torch.cat(outs, 1).cpu().numpy()
    def encode_many(self, pils):
        r = [self.encode_pils(pils[i:i + self.bs]) for i in range(0, len(pils), self.bs)]
        return np.concatenate(r, 0) if r else np.zeros((0, 1), np.float32)

class ReIDHead(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.pre = nn.Sequential(nn.LayerNorm(d_in), nn.Linear(d_in, 2048), nn.GELU(), nn.Dropout(0.2), nn.Linear(2048, d_out))
        self.bn = nn.BatchNorm1d(d_out)
    def embed(self, x): return F.normalize(self.bn(self.pre(x)), dim=1)
class VehHead(nn.Module):
    def __init__(self, d, c, h=1024, p=0.3):
        super().__init__(); self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, c))
    def forward(self, x): return self.net(x)
class CRNN(nn.Module):
    def __init__(self, nc):
        super().__init__()
        def blk(i, o, pool): return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True), *([nn.MaxPool2d(pool)] if pool else []))
        self.cnn = nn.Sequential(blk(3, 64, (2, 2)), blk(64, 128, (2, 2)), blk(128, 256, None), blk(256, 256, (2, 1)), blk(256, 384, None), blk(384, 384, (2, 1)))
        self.rnn = nn.LSTM(384, 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2); self.fc = nn.Linear(512, nc)
    def forward(self, x):
        f = self.cnn(x).mean(2).permute(0, 2, 1); o, _ = self.rnn(f); return self.fc(o)

def _ctc_decode(logits, charset):
    p = logits.softmax(-1); conf, idx = p.max(-1); res = []
    for b in range(idx.shape[0]):
        s, cs, prev = [], [], 0
        for t in range(idx.shape[1]):
            k = int(idx[b, t])
            if k != 0 and k != prev: s.append(charset[k]); cs.append(float(conf[b, t]))
            prev = k
        res.append((''.join(s), float(np.mean(cs)) if cs else 0.0))
    return res

class YemenVisionStage9:
    def __init__(self, final_dir, device=None, zoo_dir=None):
        self.dir = Path(final_dir); self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        zoo = Path(zoo_dir) if zoo_dir else self.dir.parent.parent / 'model_zoo'
        os.environ.setdefault('HF_HOME', str(zoo / 'hf')); os.environ.setdefault('TORCH_HOME', str(zoo / 'torch')); os.environ.setdefault('OPEN_CLIP_CACHE_DIR', str(zoo / 'hf'))
        self.cfg = json.loads((self.dir / 'config.json').read_text(encoding='utf-8'))
        self.enc = GiantEncoder(self.cfg['backbones'], self.device)
        self.reid = None; self.veh = None; self.ocr = None; self.plate_det = None; self.color = None
        if (self.dir / 'reid_head.pt').is_file():
            a = torch.load(self.dir / 'reid_head.pt', map_location='cpu'); self.reid = ReIDHead(a['in_dim'], a['out_dim']); self.reid.load_state_dict(a['state'], strict=False); self.reid.to(self.device).eval()
        if (self.dir / 'vehicle_class_head.pt').is_file():
            a = torch.load(self.dir / 'vehicle_class_head.pt', map_location='cpu'); self.veh_meta = a
            self.veh = VehHead(a['in_dim'], len(a['classes'])); self.veh.load_state_dict(a['state']); self.veh.to(self.device).eval()
        if (self.dir / 'plate_ocr_crnn.pt').is_file():
            a = torch.load(self.dir / 'plate_ocr_crnn.pt', map_location='cpu'); self.ocr_meta = a
            self.ocr = CRNN(len(a['charset'])); self.ocr.load_state_dict(a['state']); self.ocr.to(self.device).eval()
        if (self.dir / 'plate_detector.pt').is_file():
            from ultralytics import YOLO
            self.plate_det = YOLO(str(self.dir / 'plate_detector.pt'))
        if (self.dir / 'plate_color_model.json').is_file(): self.color = json.loads((self.dir / 'plate_color_model.json').read_text(encoding='utf-8'))
        self._easy = None

    @torch.no_grad()
    def embed_and_classify(self, pils):
        Fm = self.enc.encode_many(pils); out = {'feats': Fm}
        t = torch.from_numpy(Fm).to(self.device).float()
        out['emb'] = self.reid.embed(t).cpu().numpy() if self.reid is not None else Fm / (np.linalg.norm(Fm, axis=1, keepdims=True) + 1e-9)
        if self.veh is not None:
            p = F.softmax(self.veh(t) / self.veh_meta['temperature'], 1).cpu().numpy(); out['class_probs'] = p; res = []
            for r in p:
                k = int(r.argmax()); ans = r[k] >= self.veh_meta['reject_threshold']
                res.append({'fine': self.veh_meta['classes'][k] if ans else 'unknown', 'coarse': self.veh_meta['coarse_of'][self.veh_meta['classes'][k]] if ans else None, 'conf': float(r[k])})
            out['cls'] = res
        return out

    def _color_cat(self, crop):
        if not self.color: return 'unknown'
        import cv2
        a = np.asarray(crop.convert('RGB').resize((96, 24)), np.uint8); lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
        km = cv2.kmeans(lab, 2, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 15, 0.5), 2, cv2.KMEANS_PP_CENTERS)
        f = km[2][np.bincount(km[1].ravel()).argmax()]; X = np.asarray(self.color['X'], np.float32); d = np.linalg.norm(X - f, axis=1)
        idx = np.argsort(d)[:self.color['k']]; return collections.Counter(self.color['Y'][i] for i in idx).most_common(1)[0][0]

    def _read(self, crop):
        if self.ocr is not None:
            m = self.ocr_meta; x = torch.from_numpy(np.asarray(crop.convert('RGB').resize((m['W'], m['H']), Image.BICUBIC), np.float32) / 255.).permute(2, 0, 1)[None].to(self.device)
            with torch.no_grad(): t, c = _ctc_decode(self.ocr(x).float().cpu(), m['charset'])[0]
            if t: return t, c
        try:
            if self._easy is None:
                import easyocr; self._easy = easyocr.Reader(self.cfg.get('plate_langs', ['ar', 'en']), gpu=(self.device == 'cuda'), verbose=False)
            rr = self._easy.readtext(np.asarray(crop.convert('RGB')), detail=1)
            if rr: return ' '.join(x[1] for x in rr), float(np.mean([x[2] for x in rr]))
        except Exception: pass
        return '', 0.0

    def read_plates(self, vehicle_pils, max_frames=10):
        if self.plate_det is None: return {'vote': vote_plate([]), 'parsed': None, 'plate_category': 'unknown', 'n_reads': 0}
        reads, cats = [], []
        for im in sorted(vehicle_pils, key=lambda i: -i.size[0] * i.size[1])[:max_frames]:
            r = self.plate_det.predict(np.asarray(im.convert('RGB')), conf=0.25, imgsz=self.cfg.get('plate_det_imgsz', 960), verbose=False)[0]
            for b in r.boxes[:1]:
                x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]; pc = im.crop((x1, y1, x2, y2)); t, c = self._read(pc)
                if t: reads.append((t, c * float(b.conf[0]))); cats.append(self._color_cat(pc))
        v = vote_plate(reads); gov = self.cfg['governorates']
        return {'vote': v, 'parsed': parse_plate(v['text'], gov) if v['text'] else None, 'plate_category': collections.Counter(cats).most_common(1)[0][0] if cats else 'unknown', 'n_reads': len(reads)}

    def summarize(self, vehicle_pils, quals=None, cam=None, t0=0.0, t1=0.0, with_plate=True):
        o = self.embed_and_classify(vehicle_pils); q = quals if quals is not None else [im.size[0] * im.size[1] for im in vehicle_pils]
        s = {'cam': cam, 't0': t0, 't1': t1, 'emb': quality_weighted_mean(o['emb'], q)}
        if 'class_probs' in o:
            p = o['class_probs'].mean(0); k = int(p.argmax()); ans = p[k] >= self.veh_meta['reject_threshold']; c = self.veh_meta['classes'][k]
            s.update({'fine': c if ans else 'unknown', 'coarse': self.veh_meta['coarse_of'][c] if ans else None, 'cls_conf': float(p[k]), 'top3': [(self.veh_meta['classes'][i], float(p[i])) for i in np.argsort(-p)[:3]]})
        if with_plate:
            pa = self.read_plates(vehicle_pils); s.update({'plate': pa['vote']['text'] or None, 'plate_conf': pa['vote']['conf'], 'plate_info': pa['parsed'], 'plate_category': pa['plate_category']})
        return s

    def associate(self, tracklets):
        topo = {tuple(k.split('|')): tuple(v) for k, v in self.cfg.get('camera_topology', {}).items()}
        return associate_tracklets(tracklets, self.cfg['mtmc'], topo)
"""
RUNTIME_SRC = RUNTIME_HEAD + COMMON_SRC + RUNTIME_BODY
compile(RUNTIME_SRC, 'yemenvision_stage9_runtime.py', 'exec')     # يفشل هنا فوراً إن كان فيه خطأ صياغة
status('ok', f'وحدة التشغيل جاهزة ({len(RUNTIME_SRC.splitlines())} سطر)')

✅ وحدة التشغيل جاهزة (317 سطر)


In [22]:
# ١٣-ب. بوابات الترقية + كتابة النموذج النهائي في مجلد منفصل على Drive
GATES_CFG = {'vehicle_top1_lower_min': 0.70, 'plate_exact_lower_min': 0.80, 'reid_requires_beating_baseline': True}
SAVE_BACKBONE_WEIGHTS = False     # True ينسخ أوزان الهياكل العملاقة (عشرات GB) — عادةً لا حاجة، تبقى في model_zoo
FORCE_PROMOTE = False             # True يتجاوز البوابات (غير مُوصى به)

gates = {}
yr = REID_REPORT.get('yemen') if isinstance(REID_REPORT, dict) else None
if isinstance(yr, dict) and isinstance(yr.get('trained_head'), dict):
    lo = yr['trained_head']['mAP_ci95'][0]; gates['reid_yemen'] = 'pass' if lo >= yr['baseline_raw_giant']['mAP'] else 'fail'
else: gates['reid_yemen'] = 'NOT_EVALUATED'
vr = load_json(RUN_DIR / 'vehicle_class_report.json')
gates['vehicle_class'] = ('pass' if vr['top1_all'][1] >= GATES_CFG['vehicle_top1_lower_min'] else 'fail') if vr else 'NOT_EVALUATED'
pr_ = load_json(RUN_DIR / 'plate_ocr_report.json')
gates['plate_ocr'] = ('pass' if pr_['crnn']['exact_match'][1] >= GATES_CFG['plate_exact_lower_min'] else 'fail') if pr_ and isinstance(pr_.get('crnn'), dict) else 'NOT_EVALUATED'
promote = FORCE_PROMOTE or (all(v in ('pass', 'NOT_EVALUATED') for v in gates.values()) and any(v == 'pass' for v in gates.values()))
print('البوابات:', gates, '| ترقية إلى latest:', promote)

FINAL_DIR = FINAL_ROOT / f'stage9_{RUN_ID}'; FINAL_DIR.mkdir(parents=True, exist_ok=True)
files = []
def _save(obj, name): p = FINAL_DIR / name; torch.save(obj, str(p) + '.tmp'); os.replace(str(p) + '.tmp', p); files.append(p)
if REID_ART: _save({'state': {k: v for k, v in REID_ART['state'].items() if not k.startswith('cls.')}, 'in_dim': REID_ART['in_dim'], 'out_dim': REID_ART['out_dim']}, 'reid_head.pt')
if VEH_ART: _save({k: VEH_ART[k] for k in ('classes', 'coarse_of', 'state', 'in_dim', 'temperature', 'reject_threshold')}, 'vehicle_class_head.pt')
if PLATE_OCR: _save({k: PLATE_OCR[k] for k in ('state', 'charset', 'H', 'W')}, 'plate_ocr_crnn.pt')
if PLATE_DET_PATH: atomic_copy(PLATE_DET_PATH, FINAL_DIR / 'plate_detector.pt'); files.append(FINAL_DIR / 'plate_detector.pt')
if PLATE_COLOR: save_json(FINAL_DIR / 'plate_color_model.json', PLATE_COLOR); files.append(FINAL_DIR / 'plate_color_model.json')
if SAVE_BACKBONE_WEIGHTS:
    for i, prt in enumerate(ENC.parts): _save({'spec': prt.spec, 'state': {k: v.cpu() for k, v in prt.m.state_dict().items()}}, f'backbone_{i}.pt')
save_json(FINAL_DIR / 'config.json', {'backbones': [p.spec for p in ENC.parts], 'mtmc': ASSOC_TUNED, 'governorates': YEMEN_GOVERNORATES_AR, 'plate_langs': CFG['plate_langs'], 'plate_det_imgsz': CFG['plate_det_imgsz'],
    'camera_topology': {'|'.join(k): list(v) for k, v in CAMERA_TOPOLOGY.items()}, 'privacy': {'hash_plates': CFG['PRIVACY']['hash_plates']}, 'vehicle_taxonomy': VEHICLE_TAXONOMY})
(FINAL_DIR / 'yemenvision_stage9_runtime.py').write_text('import numpy as np\n' + RUNTIME_SRC, encoding='utf-8')
for n in ('vehicle_class_report.json', 'reid_report.json', 'plate_ocr_report.json', 'plate_detector_val.json', 'assoc_tuning.json'):
    if (RUN_DIR / n).is_file(): atomic_copy(RUN_DIR / n, FINAL_DIR / 'reports' / n)
(FINAL_DIR / 'README.md').write_text(f"""# YemenVision Stage 9 — {RUN_ID}
الاستخدام:
    import sys; sys.path.insert(0, r'{FINAL_ROOT}/latest')
    from yemenvision_stage9_runtime import YemenVisionStage9
    yv = YemenVisionStage9(r'{FINAL_ROOT}/latest')
    s = yv.summarize([PIL_crop1, PIL_crop2, ...], cam='cam01', t0=12.0, t1=19.5)
    labels = yv.associate([s1, s2, ...])
البوابات: {json.dumps(gates, ensure_ascii=False)}
الهياكل العملاقة تُحمَّل من model_zoo/ أو الإنترنت (انظر zoo_manifest.json للرخص).
""", encoding='utf-8')
man = {'run_id': RUN_ID, 'gates': gates, 'promoted': bool(promote), 'tier': TIER, 'gpu': GPU_NAME, 'files': {p.name: {'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(FINAL_DIR.rglob('*')) if p.is_file()}}
save_json(FINAL_DIR / 'manifest.json', man)
import zipfile
zp = FINAL_ROOT / f'stage9_{RUN_ID}.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FINAL_DIR.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(FINAL_DIR))
if promote:
    latest = FINAL_ROOT / 'latest'; latest.mkdir(exist_ok=True)
    for p in FINAL_DIR.rglob('*'):
        if p.is_file(): atomic_copy(p, latest / p.relative_to(FINAL_DIR))
    status('ok', f'رُقّي إلى {latest}')
else: status('warn', 'لم يُرقَّ إلى latest (بوابة فاشلة أو لا شيء مُقاس). النموذج محفوظ في المجلد المؤرَّخ للمراجعة.')
status('ok', f'النموذج النهائي: {FINAL_DIR}\nالحزمة: {zp}')

البوابات: {'reid_yemen': 'NOT_EVALUATED', 'vehicle_class': 'NOT_EVALUATED', 'plate_ocr': 'NOT_EVALUATED'} | ترقية إلى latest: False
⚠️ لم يُرقَّ إلى latest (بوابة فاشلة أو لا شيء مُقاس). النموذج محفوظ في المجلد المؤرَّخ للمراجعة.
✅ النموذج النهائي: /content/drive/MyDrive/YemenVision_MTMC/final_models/stage9_20261124T192636Z
الحزمة: /content/drive/MyDrive/YemenVision_MTMC/final_models/stage9_20261124T192636Z.zip


In [23]:
# ١٣-ج. تحقق نهائي: حمّل النموذج المصدَّر من Drive كأنك في جلسة جديدة وجرّبه
import importlib
final_for_test = (FINAL_ROOT / 'latest') if (FINAL_ROOT / 'latest' / 'config.json').is_file() else FINAL_DIR
sys.path.insert(0, str(final_for_test)); import yemenvision_stage9_runtime as yvr; importlib.reload(yvr)
yv = yvr.YemenVisionStage9(final_for_test)
sample = list_images(VEH_CROPS)[:4] if VEH_CROPS.exists() else []
if sample:
    pils = [_load(p) for p in sample]; o = yv.embed_and_classify(pils)
    print('شكل البصمات:', o['emb'].shape, '| أول فئة متوقعة:', (o.get('cls') or [None])[0])
    assert np.allclose(np.linalg.norm(o['emb'], axis=1), 1, atol=1e-3)
    status('ok', 'النموذج المصدَّر يعمل من ملفه المنفصل')
else: status('pause', 'لا صور للتجربة السريعة؛ حمّل النموذج بنجاح لكن لم يُختبر على صور')

Using cache found in /content/drive/MyDrive/YemenVision_MTMC/model_zoo/torch/hub/facebookresearch_dinov2_main


⏸️ لا صور للتجربة السريعة؛ حمّل النموذج بنجاح لكن لم يُختبر على صور


## التشغيل المقصود

افتح الدفتر في Colab، فعّل GPU، ثم شغّل الخلايا بالترتيب. لا تُشغّل خلايا التدريب محلياً؛ ستُنسخ البيانات تلقائياً إلى `/content/stage9_work/data` وتُحفظ النماذج والتقارير على Drive.

## التشغيل المقصود

افتح الدفتر في Colab، فعّل GPU، ثم شغّل الخلايا بالترتيب. لا تُشغّل خلايا التدريب محلياً؛ ستُنسخ البيانات تلقائياً إلى `/content/stage9_work/data` وتُحفظ النماذج والتقارير على Drive.

## ما بعد التشغيل: كيف ترفع الدقة فعلاً
| الهدف | الرافع الأكبر (بالترتيب) |
|---|---|
| فئات دقيقة (تاكسي/هايلكس/دايو) | ≥100 مقصوصة لكل فئة من **كاميراتك** وبعدة زوايا وإضاءات؛ ثم راجع `top_confusions` وأضف أمثلة للأزواج الملتبسة؛ ثم ارفع الطبقة (TIER) |
| إعادة التعرف بأشكال مختلفة | مجلدات `yemen_reid_ids/<id>/c<cam>_*.jpg` لنفس المركبة من ≥2 كاميرا وزوايا متباينة (أمام/خلف/جانب). هذا أهم من أي هيكل أكبر |
| اللوحات | ≥1000 قصاصة لوحة منسوخة النص بدقة، تشمل الليل والمطر والانحراف؛ و`plates/categories` لفئة اللوحة |
| MTMC | زوّد `CAMERA_TOPOLOGY` بأزمنة السفر الحقيقية، وعدّل `CAM_CLOCK_OFFSETS`، وضع `mtmc_gt/` لقياس IDF1 الحقيقي |

**تحذيرات:** (١) الأرقام على بياناتك أنت هي الحَكَم؛ نتائج VeRi لا تثبت العمل على طرق اليمن. (٢) راجع الرخص قبل الاستخدام التجاري. (٣) اللوحات بيانات شخصية: أبقِ `hash_plates=True` وقيّد الوصول إلى السجلات.